# Phase 5 -- Legal Statute Identification: all runs in one notebook

A document -> the **set** of statute ids that apply (multi-label). Metric: **macro-F1 over statutes**, so a
statute seen 30 times counts as much as one seen 3,000 times.

| Run | Phase-1 model | Technique | Why this pairing | Submit |
|---|---|---|---|---|
| **S1** | D encoder (100M) | **LoRMA** (rank 16, every attention + MLP layer) | best backbone + best method in Phase 4 (0.7693 official); compulsory | yes |
| **S2** | B (122M) | **QLoRA** (frozen 4-bit NF4 copy of B + LoRA rank 16) | B did well with low-rank updates in Phase 4 (0.7524 dev); tests whether 4 bits keep that | yes |
| **S3** | C (113M) | **Houlsby adapters** (bottleneck 64 after attention and after MLP + norms) | C is the other bidirectional reader; adapters add capacity *between* layers instead of inside them | yes |
| **S4** | A (163M) | **Prefix tuning** (16 learned states at every layer, started from real activations) | prefix tuning was designed for left-to-right decoders (GPT-2) -- A is our pure decoder | yes |
| **S5** | D encoder | **Partial / layer-wise fine-tuning** (top 6 of 12 layers, layer-wise LR decay) | in Phase 4 full fine-tuning of D lost to LoRMA; does training only the top half fix that? | yes |
| **S6** | half of D (62M: layers 0,2,4,..,10) | **Knowledge distillation** from S1 + S3 (213M) | DistilBERT-style student; learns gold statutes + teachers' statute probabilities | yes |
| **S7** | -- | **Ensemble** of S1-S6 (mean probability, own thresholds) | built here, no GPU | yes |
| R1 | D encoder | full fine-tuning | reference: does parameter-efficient training keep full fine-tuning's score? | dev only |
| R2 | same student as S6 | gold labels only | control: what does distillation add? | dev only |
| TF-IDF | none | word 1-2-grams + logistic regression per statute | does Phase-1 pretraining help at all? (CPU, runs alongside) | dev only |

**Pipeline (every run except the student)** -- documents are cut into 512-token chunks (64-token overlap; at most 12
per document, spread evenly).
1. **Adapt** the Phase-1 model with its technique, topped by a **label-wise attention head** (each statute has its
   own attention over the chunk's tokens, as in CAML/LAAT). A document is a bag of up to 3 random chunks; its statute
   scores are the **log-sum-exp pool** of the chunk scores (a statute applies if *some* part of the document
   supports it). Weighted BCE (rare statutes weigh more). 80% of train.
2. **Encode** every chunk of the held-out 20% of train, dev and test.
3. **Read the whole document**: 6 small readers over the chunk vectors -- mean, max, attention, **label-wise
   attention** (each statute looks for its own evidence), label-wise attention **+ statute co-occurrence**,
   transformer + label-wise attention -- plus a **loss study** (BCE / weighted BCE / focal / asymmetric) and a
   **section study** (first chunk, first / middle / last third). Trained on the held-out 20%, chosen by
   validation macro-average-precision; dev stays clean.
4. **Thresholds** on dev: 0.5, one global threshold, one per statute, or per statute shrunk toward the global one
   -- the strategy is picked by an honest 2-fold dev check. Every document gets at least its best statute.

**How to run**
1. Inputs: `legallm-tokenizers` (`t1_bpe.json`, `t2_unigram.model`) and the datasets holding the **A, B, C and D**
   checkpoints (file names start `A_step...`, `B_step...`, `C_step...`, `D_step...`; `.pt`, `.zip` or a folder Kaggle
   unzipped are all fine).
2. Secret on: `HF`. Accelerator **GPU T4 x2**, **internet on**.
3. **Save Version -> Save & Run All.** Time: about 7-8.5 h (two runs at a time; every run has a time budget, and the
   scheduler skips a run that would not finish before 11 h).
4. Submit `p5_out/S1 ... S7/submission.zip` (7 files). Download `p5_out/report_files.zip` (all tables, for the report).

## 1. Inputs: tokenizers and checkpoints

In [ ]:
import csv
import gc
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import threading
import time
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import torch

NB_T0 = time.time()
N_GPUS = torch.cuda.device_count()
print("PyTorch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPUS)])
KAGGLE_INPUT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
OUT_DIR = WORK_DIR / "p5_out"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR = WORK_DIR / "ckpts"
CKPT_DIR.mkdir(exist_ok=True)


def step_of(p):
    return int(Path(p).name.split("_step")[1].split("_")[0].split(".")[0])


def _rebuild_from_folder(folder, dest):
    """Kaggle unzips an uploaded checkpoint (a .pt file is a zip archive inside),
    leaving a folder with data.pkl, data/0, version, ... Re-pack it into a .pt."""
    pkl = next(Path(folder).rglob("data.pkl"))
    root = pkl.parent
    with zipfile.ZipFile(dest, "w", zipfile.ZIP_STORED) as zf:
        for f in sorted(root.rglob("*")):
            if f.is_file():
                zf.write(f, arcname="archive/" + f.relative_to(root).as_posix())
    return dest


def _collect(prefix):
    """{name: path} of usable <prefix>_step*.pt checkpoints under /kaggle/input:
    a .pt file; a .zip holding the .pt; the .pt renamed .zip; or a folder that
    Kaggle unzipped from an uploaded checkpoint (re-packed here)."""
    found = {}
    if not KAGGLE_INPUT.exists():
        return found
    for p in KAGGLE_INPUT.rglob(f"{prefix}_step*"):
        stem = p.name[:-3] if p.name.endswith(".pt") else (p.name[:-4] if p.name.endswith(".zip") else p.name)
        dest = CKPT_DIR / (stem + ".pt")
        if p.is_file() and p.suffix == ".pt":
            found.setdefault(p.name, p)
        elif p.is_dir() and any(p.rglob("data.pkl")) and dest.name not in found:
            found[dest.name] = _rebuild_from_folder(p, dest)
            print(f"  re-packed unzipped checkpoint folder {p} -> {dest.name}")
    for z in KAGGLE_INPUT.rglob("*.zip"):
        try:
            with zipfile.ZipFile(z) as zf:
                names = zf.namelist()
                inner = [n for n in names if Path(n).name.startswith(f"{prefix}_step") and n.endswith(".pt")]
                for n in inner:
                    dest = CKPT_DIR / Path(n).name
                    if dest.name not in found:
                        with zf.open(n) as src, open(dest, "wb") as dst:
                            shutil.copyfileobj(src, dst)
                        found[dest.name] = dest
                if not inner and z.name.startswith(f"{prefix}_step") and any(n.endswith("data.pkl") for n in names):
                    dest = CKPT_DIR / (z.stem + ".pt")
                    if dest.name not in found:
                        shutil.copy(z, dest)
                        found[dest.name] = dest
        except zipfile.BadZipFile:
            continue
    return found




def checkpoints(prefix):
    found = _collect(prefix)
    assert found, f"No {prefix}_step* checkpoint (.pt, .zip or unzipped folder) under /kaggle/input -- attach it."
    return max(found.values(), key=step_of)


def find_one(name, label):
    hits = sorted(KAGGLE_INPUT.rglob(name)) if KAGGLE_INPUT.exists() else []
    assert hits, f"{name} not found under /kaggle/input -- attach {label}."
    return hits[0]



T1_PATH = find_one("t1_bpe.json", "legallm-tokenizers")
T2_PATH = find_one("t2_unigram.model", "legallm-tokenizers")
CKPT = {k: str(checkpoints(k)) for k in ("A", "B", "C", "D")}
for k, p in CKPT.items():
    print(f"{k}: {Path(p).name}")

## 2. Code files (written to /kaggle/working)

In [ ]:
%%writefile /kaggle/working/legal_d.py
"""legal_d.py -- Model D: encoder-decoder legal language model + its trainer.

Model D (T5 / RedLLM-style encoder-decoder, tokenizer T2):
  * Encoder: bidirectional self-attention over the prompt.
  * Decoder: causal self-attention over the continuation + cross-attention
    to the encoder output.
  * Shared (tied) embedding for encoder input, decoder input and output head.
  * RMSNorm (pre-norm), RoPE in self-attention, QK-norm, GQA, SwiGLU, no biases.

Trainer: one process per GPU (DDP via torch.distributed.run), Muon optimizer
for the hidden weight matrices + AdamW for embeddings/norms, fp16 autocast
with GradScaler (T4 has no bf16), time-based warmup-stable-decay LR
schedule (so the LR always finishes its decay when the time budget ends),
flat checkpoints with resume, periodic push to a Kaggle dataset.

Run training:   python -m torch.distributed.run --standalone --nproc_per_node=2 legal_d.py d_config.json
Import helpers: import legal_d  (model, generation, scoring)
"""
import contextlib
import datetime
import itertools
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
import zipfile
from collections import Counter
from pathlib import Path

import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint

ARCH_ID = "D"

MODEL_D_CONFIG = dict(
    vocab_size=32_000,        # T2 (SentencePiece Unigram), same as C
    d_model=768,
    n_enc_layer=12,
    n_dec_layer=12,
    n_head=12,
    n_kv_head=4,              # GQA 3:1 (self- and cross-attention)
    d_head=64,
    d_ff=2048,                # SwiGLU (same FFN parameter count as C's 4x GELU)
    max_enc_len=1024,
    max_dec_len=1024,
    qk_norm=True,
    mlp="swiglu",
    rope_theta=10_000.0,
    tokenizer_id="T2",
    arch="encoder-decoder",
    tied_embeddings=True,
)

# ---------------------------------------------------------------------------
# Tokenizer (T2)
# ---------------------------------------------------------------------------
sp = None
EOS_ID = None
DEC_START_ID = None
BANNED_IN_OUTPUT = []


def load_tokenizer(path):
    global sp, EOS_ID, DEC_START_ID, BANNED_IN_OUTPUT
    import sentencepiece as spm
    sp = spm.SentencePieceProcessor()
    sp.load(str(path))
    EOS_ID = sp.eos_id()
    assert EOS_ID >= 0, "T2 tokenizer has no <eos>"
    DEC_START_ID = sp.bos_id() if sp.bos_id() >= 0 else EOS_ID      # decoder start token
    BANNED_IN_OUTPUT = sorted({sp.pad_id(), sp.unk_id(), sp.bos_id(), sp.eos_id()} - {-1})
    return sp


def encode(text):
    return sp.encode(text, out_type=int)


def decode(ids):
    return sp.decode(ids)


# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        x32 = x.float()
        return (x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight.float()).to(x.dtype)


def rope_cache(n, d_head, theta):
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_head, 2).float() / d_head))
    freqs = torch.outer(torch.arange(n).float(), inv_freq)
    return freqs.cos(), freqs.sin()                      # [n, d_head/2]


def apply_rope(x, cos, sin):
    """x [B, H, T, d]; cos/sin [T, d/2]. Rotate-half form, computed in fp32."""
    cos, sin = cos[None, None].float(), sin[None, None].float()
    half = x.shape[-1] // 2
    x1, x2 = x[..., :half].float(), x[..., half:].float()
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1).to(x.dtype)


def expand_kv(t, n_rep):
    if n_rep == 1:
        return t
    B, H, T, D = t.shape
    return t[:, :, None].expand(B, H, n_rep, T, D).reshape(B, H * n_rep, T, D)


class KVCache:
    """Decoder self-attention cache (allocated on first use)."""

    def __init__(self, max_len):
        self.max_len, self.pos, self.k, self.v = max_len, 0, None, None

    def update(self, k, v):
        if self.k is None:
            B, H, _, D = k.shape
            self.k = torch.zeros(B, H, self.max_len, D, device=k.device, dtype=k.dtype)
            self.v = torch.zeros_like(self.k)
        T = k.shape[2]
        self.k[:, :, self.pos:self.pos + T] = k
        self.v[:, :, self.pos:self.pos + T] = v
        self.pos += T
        return self.k[:, :, :self.pos], self.v[:, :, :self.pos]


class SelfAttention(nn.Module):
    def __init__(self, cfg, causal):
        super().__init__()
        d, h, kv, dh = cfg["d_model"], cfg["n_head"], cfg["n_kv_head"], cfg["d_head"]
        self.causal, self.n_head, self.n_kv_head, self.d_head = causal, h, kv, dh
        self.q_proj = nn.Linear(d, h * dh, bias=False)
        self.k_proj = nn.Linear(d, kv * dh, bias=False)
        self.v_proj = nn.Linear(d, kv * dh, bias=False)
        self.o_proj = nn.Linear(h * dh, d, bias=False)
        self.q_norm = RMSNorm(dh) if cfg["qk_norm"] else None
        self.k_norm = RMSNorm(dh) if cfg["qk_norm"] else None

    def forward(self, x, cos, sin, key_mask=None, cache=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        if self.q_norm is not None:
            q, k = self.q_norm(q), self.k_norm(k)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        first_chunk = cache is None or cache.pos == 0
        if cache is not None:
            assert T == 1 or cache.pos == 0, "cached decoding feeds one token at a time"
            k, v = cache.update(k, v)
        rep = self.n_head // self.n_kv_head
        k, v = expand_kv(k, rep), expand_kv(v, rep)
        if key_mask is not None:                                  # [B, Tk] True = real token
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=key_mask[:, None, None, :])
        else:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=self.causal and T > 1 and first_chunk)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head))


class CrossAttention(nn.Module):
    """Decoder queries attend to the encoder output (no RoPE: T5-style)."""

    def __init__(self, cfg):
        super().__init__()
        d, h, kv, dh = cfg["d_model"], cfg["n_head"], cfg["n_kv_head"], cfg["d_head"]
        self.n_head, self.n_kv_head, self.d_head = h, kv, dh
        self.q_proj = nn.Linear(d, h * dh, bias=False)
        self.k_proj = nn.Linear(d, kv * dh, bias=False)
        self.v_proj = nn.Linear(d, kv * dh, bias=False)
        self.o_proj = nn.Linear(h * dh, d, bias=False)
        self.q_norm = RMSNorm(dh) if cfg["qk_norm"] else None
        self.k_norm = RMSNorm(dh) if cfg["qk_norm"] else None

    def project_kv(self, enc_out):
        B, S, _ = enc_out.shape
        k = self.k_proj(enc_out).view(B, S, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(enc_out).view(B, S, self.n_kv_head, self.d_head).transpose(1, 2)
        if self.k_norm is not None:
            k = self.k_norm(k)
        return k, v

    def forward(self, x, enc_kv, enc_mask=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        if self.q_norm is not None:
            q = self.q_norm(q)
        rep = self.n_head // self.n_kv_head
        k, v = expand_kv(enc_kv[0], rep), expand_kv(enc_kv[1], rep)
        mask = enc_mask[:, None, None, :] if enc_mask is not None else None
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head))


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, f = cfg["d_model"], cfg["d_ff"]
        self.gate_proj = nn.Linear(d, f, bias=False)
        self.up_proj = nn.Linear(d, f, bias=False)
        self.down_proj = nn.Linear(f, d, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg["d_model"])
        self.attn = SelfAttention(cfg, causal=False)
        self.mlp_norm = RMSNorm(cfg["d_model"])
        self.mlp = SwiGLU(cfg)

    def forward(self, x, cos, sin, key_mask=None):
        x = x + self.attn(self.attn_norm(x), cos, sin, key_mask)
        return x + self.mlp(self.mlp_norm(x))


class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.self_norm = RMSNorm(cfg["d_model"])
        self.self_attn = SelfAttention(cfg, causal=True)
        self.cross_norm = RMSNorm(cfg["d_model"])
        self.cross_attn = CrossAttention(cfg)
        self.mlp_norm = RMSNorm(cfg["d_model"])
        self.mlp = SwiGLU(cfg)

    def forward(self, x, cos, sin, enc_out=None, enc_mask=None, enc_kv=None, cache=None):
        if enc_kv is None:
            enc_kv = self.cross_attn.project_kv(enc_out)
        x = x + self.self_attn(self.self_norm(x), cos, sin, cache=cache)
        x = x + self.cross_attn(self.cross_norm(x), enc_kv, enc_mask)
        return x + self.mlp(self.mlp_norm(x))


def _ce_chunk(h, w, t):
    logits = F.linear(h, w).float()
    lse = torch.logsumexp(logits, dim=-1)
    valid = (t != -100).float()
    nll = (lse - logits.gather(-1, t.clamp(min=0)[:, None]).squeeze(-1)) * valid
    return torch.stack([nll.sum(), (lse.pow(2) * valid).sum()])


def chunked_loss(h, w, targets, chunk=4096):
    """[1, 3] = [sum of token NLL, number of scored tokens, sum of logsumexp^2].
    Logits are built chunk by chunk (and recomputed in backward), so the full
    [tokens x 32k] logit tensor is never stored."""
    h = h.reshape(-1, h.shape[-1])
    t = targets.reshape(-1)
    parts = []
    for s in range(0, h.shape[0], chunk):
        hs, ts = h[s:s + chunk], t[s:s + chunk]
        if torch.is_grad_enabled() and h.requires_grad:
            parts.append(grad_checkpoint(_ce_chunk, hs, w, ts, use_reentrant=False))
        else:
            parts.append(_ce_chunk(hs, w, ts))
    tot = torch.stack(parts).sum(0)
    count = (t != -100).sum().to(tot.dtype)
    return torch.stack([tot[0], count, tot[1]]).unsqueeze(0)


class EncoderDecoderLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        d, V = config["d_model"], config["vocab_size"]
        self.embed = nn.Embedding(V, d)
        self.enc_blocks = nn.ModuleList([EncoderBlock(config) for _ in range(config["n_enc_layer"])])
        self.enc_norm = RMSNorm(d)
        self.dec_blocks = nn.ModuleList([DecoderBlock(config) for _ in range(config["n_dec_layer"])])
        self.dec_norm = RMSNorm(d)
        self.head = nn.Linear(d, V, bias=False)
        self.head.weight = self.embed.weight                  # tied
        cos, sin = rope_cache(max(config["max_enc_len"], config["max_dec_len"]), config["d_head"], config["rope_theta"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.grad_ckpt = False
        self.amp = True                                        # fp16 autocast on GPU (fp32 weights)
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)
        for name, p in self.named_parameters():                 # scaled residual-output init
            if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
                n_res = 2 * config["n_enc_layer"] if name.startswith("enc_") else 3 * config["n_dec_layer"]
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(n_res))

    def _run(self, blk, *args, **kwargs):
        if self.grad_ckpt and self.training and torch.is_grad_enabled():
            return grad_checkpoint(blk, *args, use_reentrant=False, **kwargs)
        return blk(*args, **kwargs)

    def encode(self, input_ids, attention_mask=None):
        """Contract method: encoder hidden states [B, T, d]. attention_mask
        (1 = real token) marks padding; positions start at 0 in every row
        (pad on the right)."""
        T = input_ids.shape[1]
        x = self.embed(input_ids)
        cos, sin = self.rope_cos[:T], self.rope_sin[:T]
        key_mask = attention_mask.bool() if attention_mask is not None else None
        for blk in self.enc_blocks:
            x = self._run(blk, x, cos, sin, key_mask)
        return self.enc_norm(x)

    def decode_hidden(self, dec_ids, enc_out, enc_mask=None):
        T = dec_ids.shape[1]
        x = self.embed(dec_ids)
        cos, sin = self.rope_cos[:T], self.rope_sin[:T]
        m = enc_mask.bool() if enc_mask is not None else None
        for blk in self.dec_blocks:
            x = self._run(blk, x, cos, sin, enc_out=enc_out, enc_mask=m)
        return self.dec_norm(x)

    def forward(self, enc_ids, dec_ids, targets=None, enc_mask=None):
        """No targets -> logits [B, Q, V]. With targets -> [1, 3] sums
        (NLL, count, lse^2); targets = -100 are ignored."""
        use_amp = enc_ids.is_cuda and self.embed.weight.dtype == torch.float32 and self.amp
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            enc_out = self.encode(enc_ids, enc_mask)
            h = self.decode_hidden(dec_ids, enc_out, enc_mask)
            if targets is None:
                return self.head(h)
            return chunked_loss(h, self.head.weight, targets)

    def decoder_step(self, tok, pos, caches, cross_kvs, enc_mask):
        x = self.embed(tok)
        cos, sin = self.rope_cos[pos:pos + tok.shape[1]], self.rope_sin[pos:pos + tok.shape[1]]
        for blk, cache, kv in zip(self.dec_blocks, caches, cross_kvs):
            x = blk(x, cos, sin, enc_mask=enc_mask, enc_kv=kv, cache=cache)
        return self.head(self.dec_norm(x))

    def generate(self, input_ids, max_new_tokens, kv_cache=True, sampler="greedy", **kw):
        """Contract method (one prompt, greedy): returns prompt + continuation ids."""
        assert kv_cache and sampler == "greedy"
        toks = generate_batch(self, [input_ids[0].tolist()], max_new_tokens=max_new_tokens, **kw)[0]
        return torch.tensor([input_ids[0].tolist() + toks], device=input_ids.device)


def build_model(config):
    """Contract function."""
    return EncoderDecoderLM(config)


def count_params(model):
    return sum(p.numel() for p in model.parameters())


# ---------------------------------------------------------------------------
# Generation
# ---------------------------------------------------------------------------
def _sample_top_p(logits, temperature, top_p, generator):
    probs = torch.softmax(logits / temperature, dim=-1)
    if top_p < 1.0:
        sorted_p, idx = probs.sort(dim=-1, descending=True)
        drop = (sorted_p.cumsum(-1) - sorted_p) > top_p
        sorted_p = sorted_p.masked_fill(drop, 0.0)
        probs = torch.zeros_like(probs).scatter_(-1, idx, sorted_p)
    return torch.multinomial(probs, 1, generator=generator).squeeze(-1)


@torch.no_grad()
def generate_batch(model, prompt_id_lists, max_new_tokens=288, max_enc_len=768, rep_penalty=1.0,
                   no_repeat_ngram=0, target_words=None, banned=(), vocab_limit=None, sample=False,
                   temperature=1.0, top_p=1.0, generator=None, start_id=None):
    """Continuations for several prompts at once. Encoder input is right-padded
    (masked); every decoder row starts at the same position, so no decoder
    padding is needed. Greedy by default; `sample=True` gives temperature /
    top-p sampling (used for MBR candidates). Optional repetition penalty over
    generated tokens, n-gram blocking, and early stop once every row has more
    than `target_words` words."""
    model.eval()
    device = model.embed.weight.device
    cfg = model.config
    start_id = DEC_START_ID if start_id is None else start_id
    max_enc_len = min(max_enc_len or cfg["max_enc_len"], cfg["max_enc_len"])
    prompts = [(p[-max_enc_len:] if p else [EOS_ID if EOS_ID is not None else 0]) for p in prompt_id_lists]
    B, P = len(prompts), max(len(p) for p in prompts)
    enc_ids = torch.zeros(B, P, dtype=torch.long, device=device)
    enc_mask = torch.zeros(B, P, dtype=torch.bool, device=device)
    for b, p in enumerate(prompts):
        enc_ids[b, :len(p)] = torch.tensor(p, device=device)
        enc_mask[b, :len(p)] = True
    use_amp = device.type == "cuda" and model.embed.weight.dtype == torch.float32 and getattr(model, "amp", True)
    amp = torch.autocast(device_type="cuda", dtype=torch.float16) if use_amp else contextlib.nullcontext()
    with amp:
        enc_out = model.encode(enc_ids, enc_mask)
        cross_kvs = [blk.cross_attn.project_kv(enc_out) for blk in model.dec_blocks]
        caches = [KVCache(max_new_tokens + 1) for _ in model.dec_blocks]
        tok = torch.full((B, 1), start_id, dtype=torch.long, device=device)
        logits = model.decoder_step(tok, 0, caches, cross_kvs, enc_mask)[:, -1, :].float()
    generated = [[] for _ in range(B)]
    seen = [dict() for _ in range(B)]
    for t in range(max_new_tokens):
        if banned:
            logits[:, list(banned)] = -float("inf")
        if vocab_limit is not None:
            logits[:, vocab_limit:] = -float("inf")
        if rep_penalty != 1.0 and t > 0:
            prev = torch.tensor(generated, device=device)
            sc = logits.gather(1, prev)
            logits.scatter_(1, prev, torch.where(sc > 0, sc / rep_penalty, sc * rep_penalty))
        if no_repeat_ngram and t >= no_repeat_ngram - 1:
            n = no_repeat_ngram
            for b in range(B):
                bad = seen[b].get(tuple(generated[b][-(n - 1):]) if n > 1 else ())
                if bad:
                    logits[b, list(bad)] = -float("inf")
        nxt = _sample_top_p(logits, temperature, top_p, generator) if sample else logits.argmax(dim=-1)
        for b, tk in enumerate(nxt.tolist()):
            generated[b].append(tk)
            if no_repeat_ngram and len(generated[b]) >= no_repeat_ngram:
                g = generated[b][-no_repeat_ngram:]
                seen[b].setdefault(tuple(g[:-1]), set()).add(g[-1])
        if t == max_new_tokens - 1:
            break
        if target_words and t >= 120 and t % 16 == 0 and all(len(decode(g).split()) > target_words for g in generated):
            break
        with amp:
            logits = model.decoder_step(nxt[:, None], t + 1, caches, cross_kvs, enc_mask)[:, -1, :].float()
    return generated


# ---------------------------------------------------------------------------
# Scoring helpers (mirror evaluate_phase1.py; BERTScore uses roberta-large)
# ---------------------------------------------------------------------------
def lcs_length(a, b):
    if not a or not b:
        return 0
    if len(a) < len(b):
        a, b = b, a
    masks = {}
    for j, t in enumerate(a):
        masks[t] = masks.get(t, 0) | (1 << j)
    state = 0
    for t in b:
        m = masks.get(t, 0)
        x = m | state
        y = (state << 1) | 1
        state = x & ~(x - y)
    return bin(state).count("1")


def rouge_l_f1(p_text, r_text):
    pt, rt = p_text.split(), r_text.split()
    if not pt or not rt:
        return 0.0
    n = lcs_length(pt, rt)
    if not n:
        return 0.0
    pr, rc = n / len(pt), n / len(rt)
    return 2 * pr * rc / (pr + rc)


def _ngrams(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)) if len(tokens) >= n else Counter()


def bleu4(preds, refs):
    clipped, totals, pl, rl = [0] * 4, [0] * 4, 0, 0
    for p, r in zip(preds, refs):
        pl += len(p)
        rl += len(r)
        for n in range(1, 5):
            pn, rn = _ngrams(p, n), _ngrams(r, n)
            totals[n - 1] += sum(pn.values())
            for g, c in pn.items():
                clipped[n - 1] += min(c, rn.get(g, 0))
    prec = [clipped[i] / totals[i] if totals[i] else 0.0 for i in range(4)]
    if any(x <= 0 for x in prec) or pl == 0:
        return 0.0
    return (1.0 if pl > rl else math.exp(1.0 - rl / pl)) * math.exp(sum(math.log(x) for x in prec) / 4)


def score_predictions(pred, truth, ids):
    try:
        import bert_score
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bert-score"], check=True)
        import bert_score
    hyps = [pred[i]["text"].strip() for i in ids]
    refs = [truth[i]["text"].strip() for i in ids]
    _, _, f1 = bert_score.score(hyps, refs, lang="en", batch_size=8, verbose=False,
                                device="cuda" if torch.cuda.is_available() else "cpu")
    return {"bertscore": float(f1.mean()),
            "rouge_l": sum(rouge_l_f1(h, r) for h, r in zip(hyps, refs)) / len(refs),
            "bleu4": bleu4([h.split() for h in hyps], [r.split() for r in refs])}


def to_words(toks, n_words=128):
    words = decode(toks).split()[:n_words]
    return " ".join(words) if words else "."


def predict(model, prompt_dict, ids, batch_size=16, desc="predict", **settings):
    """Greedy/sampled continuations -> {id: {"text": 128 words}}. Prompts are
    sorted by length so each batch has similar lengths."""
    from tqdm.auto import tqdm
    enc = {i: encode(prompt_dict[i]["text"]) for i in ids}
    order = sorted(ids, key=lambda i: len(enc[i]))
    out = {}
    for s in tqdm(range(0, len(order), batch_size), desc=desc, dynamic_ncols=True, mininterval=30):
        chunk = order[s:s + batch_size]
        for i, toks in zip(chunk, generate_batch(model, [enc[i] for i in chunk], **settings)):
            out[i] = {"text": to_words(toks)}
    return {i: out[i] for i in ids}


def mbr_pick(candidates):
    """Minimum-Bayes-risk choice: the candidate with the highest average
    ROUGE-L F1 against the other candidates (a consensus pick; no outside
    model involved)."""
    if len(candidates) <= 2:
        return candidates[0]
    best, best_u = 0, -1.0
    for i, c in enumerate(candidates):
        u = sum(rouge_l_f1(c, o) for j, o in enumerate(candidates) if j != i) / (len(candidates) - 1)
        if u > best_u:
            best, best_u = i, u
    return candidates[best]


def predict_mbr(model, prompt_dict, ids, n_samples=7, batch_prompts=8, greedy_settings=None,
                sample_settings=None, seed=1234, desc="mbr"):
    """For each prompt: 1 greedy candidate + n_samples sampled candidates ->
    MBR pick. Candidates of `batch_prompts` prompts are generated together."""
    from tqdm.auto import tqdm
    device = model.embed.weight.device
    gen = torch.Generator(device=device)
    gen.manual_seed(seed)
    enc = {i: encode(prompt_dict[i]["text"]) for i in ids}
    order = sorted(ids, key=lambda i: len(enc[i]))
    out = {}
    for s in tqdm(range(0, len(order), batch_prompts), desc=desc, dynamic_ncols=True, mininterval=30):
        chunk = order[s:s + batch_prompts]
        greedy = generate_batch(model, [enc[i] for i in chunk], **greedy_settings)
        sampled = generate_batch(model, [enc[i] for i in chunk for _ in range(n_samples)], sample=True,
                                 generator=gen, **sample_settings)
        for k, i in enumerate(chunk):
            cands = [to_words(greedy[k])] + [to_words(t) for t in sampled[k * n_samples:(k + 1) * n_samples]]
            out[i] = {"text": mbr_pick(cands)}
    return {i: out[i] for i in ids}


@torch.no_grad()
def dev_loss_bpb(model, dev_prompts, dev_truth, ids, max_enc_len=768, max_dec_len=512):
    """Continuation loss per token and bits-per-byte (tokenizer-independent)."""
    model.eval()
    device = model.embed.weight.device
    nll, n_tok, n_bytes = 0.0, 0, 0
    for i in ids:
        cont_text = " " + dev_truth[i]["text"].strip()
        cont = encode(cont_text)[:max_dec_len]
        ctx = encode(dev_prompts[i]["text"])[-max_enc_len:]
        if not cont or not ctx:
            continue
        enc_ids = torch.tensor([ctx], device=device)
        dec_in = torch.tensor([[DEC_START_ID] + cont[:-1]], device=device)
        tgt = torch.tensor([cont], device=device)
        out = model(enc_ids, dec_in, targets=tgt)[0]
        nll += out[0].item()
        n_tok += len(cont)
        n_bytes += len(cont_text.encode("utf-8"))
    return nll / max(1, n_tok), nll / (math.log(2) * max(1, n_bytes))


def self_test(device):
    """Checks: cached decoding == full forward; decoder is causal; encoder is
    two-way; encoder padding changes nothing; loss sums are correct; batched
    generation == one-at-a-time; Muon orthogonalises."""
    torch.manual_seed(0)
    tiny = dict(MODEL_D_CONFIG, vocab_size=101, d_model=64, n_enc_layer=2, n_dec_layer=2, n_head=4,
                n_kv_head=2, d_head=16, d_ff=96, max_enc_len=64, max_dec_len=64)
    m = build_model(tiny).to(device).eval()
    m.amp = False                                          # exactness checks in fp32 (GPU kernels still used)
    enc = torch.randint(4, 101, (2, 20), device=device)
    dec = torch.randint(4, 101, (2, 12), device=device)
    with torch.no_grad():
        full = m(enc, dec).float()
        enc_out = m.encode(enc)
        kvs = [blk.cross_attn.project_kv(enc_out) for blk in m.dec_blocks]
        caches = [KVCache(12) for _ in m.dec_blocks]
        steps = [m.decoder_step(dec[:, t:t + 1], t, caches, kvs, None) for t in range(12)]
        diff = (full - torch.cat(steps, 1).float()).abs().max().item()
        dec2 = dec.clone()
        dec2[:, 8] = (dec[:, 8] + 1) % 97 + 4
        leak = (m(enc, dec2)[:, :8] - full[:, :8]).abs().max().item()
        enc2 = enc.clone()
        enc2[:, 15] = (enc[:, 15] + 1) % 97 + 4
        two_way = (m.encode(enc2)[:, 2] - enc_out[:, 2]).abs().max().item()
        padded = torch.cat([enc, torch.zeros(2, 5, dtype=torch.long, device=device)], 1)
        mask = torch.cat([torch.ones(2, 20), torch.zeros(2, 5)], 1).to(device)
        pad_diff = (m(padded, dec, enc_mask=mask) - full).abs().max().item()
        tgt = torch.randint(4, 101, (2, 12), device=device)
        tgt[0, 3] = -100
        s = m(enc, dec, targets=tgt)[0]
        ref = F.cross_entropy(full.reshape(-1, 101), tgt.reshape(-1), ignore_index=-100, reduction="sum")
    assert diff < 1e-3, f"cached decoding != full forward ({diff})"
    assert leak < 1e-5, f"decoder sees future tokens ({leak})"
    assert two_way > 1e-5, "encoder is not bidirectional"
    assert pad_diff < 1e-4, f"encoder padding changes the output ({pad_diff})"
    assert abs(s[0].item() - ref.item()) < 1e-3 and int(s[1].item()) == 23, "loss sums wrong"
    pa, pb = enc[0].tolist(), enc[1, :9].tolist()
    both = generate_batch(m, [pa, pb], max_new_tokens=8, start_id=1)
    alone = [generate_batch(m, [pa], max_new_tokens=8, start_id=1)[0],
             generate_batch(m, [pb], max_new_tokens=8, start_id=1)[0]]
    assert both == alone, "batched generation differs from one-at-a-time"
    g = torch.Generator(device=device)
    g.manual_seed(0)
    smp = generate_batch(m, [pa, pb], max_new_tokens=8, start_id=1, sample=True, temperature=0.8, top_p=0.9,
                         generator=g)
    assert len(smp) == 2 and all(len(x) == 8 for x in smp)
    X = newton_schulz5(torch.randn(48, 96, device=device)).float()
    sv = torch.linalg.svdvals(X)
    assert 0.3 < sv.min().item() and sv.max().item() < 1.5, f"Muon orthogonalisation off: {sv.min():.2f}..{sv.max():.2f}"
    print(f"Self-test passed: cached==full {diff:.1e}, causal decoder {leak:.1e}, two-way encoder, "
          f"padding-safe {pad_diff:.1e}, loss sums ok, batched==single, sampling ok, "
          f"Muon singular values {sv.min():.2f}..{sv.max():.2f}.")


# ---------------------------------------------------------------------------
# Muon optimizer (Jordan et al. 2024; update scale from Liu et al. 2025,
# "Muon is Scalable for LLM Training": 0.2 * sqrt(max(rows, cols)) so the
# learning rate and weight decay work on AdamW's scale).
# ---------------------------------------------------------------------------
def newton_schulz5(G, steps=5, eps=1e-7):
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G.float()
    X = X / (X.norm() + eps)                              # normalise in fp32 (tiny grads survive)
    if G.is_cuda:
        X = X.half()                                       # T4: fast fp16 matmuls, values stay <= 1
    transposed = X.shape[0] > X.shape[1]
    if transposed:
        X = X.T
    for _ in range(steps):
        A = X @ X.T
        Bm = b * A + c * (A @ A)
        X = a * X + Bm @ X
    if transposed:
        X = X.T
    return X.to(G.dtype)


class Muon(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, momentum=0.95, nesterov=True, weight_decay=0.1, ns_steps=5):
        super().__init__(params, dict(lr=lr, momentum=momentum, nesterov=nesterov,
                                      weight_decay=weight_decay, ns_steps=ns_steps))

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            lr, mom, wd = group["lr"], group["momentum"], group["weight_decay"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad.float()
                state = self.state[p]
                if "momentum_buffer" not in state:
                    state["momentum_buffer"] = torch.zeros_like(g)
                buf = state["momentum_buffer"]
                buf.mul_(mom).add_(g)
                g = g.add(buf, alpha=mom) if group["nesterov"] else buf
                u = newton_schulz5(g, steps=group["ns_steps"]).float()
                p.mul_(1.0 - lr * wd)
                p.add_(u, alpha=-lr * 0.2 * math.sqrt(max(p.shape[0], p.shape[1])))
        return None


def build_optimizers(model, kind, lr, weight_decay):
    muon_params, adam_decay, adam_no_decay = [], [], []
    for name, p in model.named_parameters():
        if p.dim() == 2 and (name.startswith("enc_blocks") or name.startswith("dec_blocks")) and kind == "muon":
            muon_params.append(p)
        elif p.dim() >= 2:
            adam_decay.append(p)                           # embedding (tied output head); all matrices for AdamW
        else:
            adam_no_decay.append(p)                        # norm weights
    adam = torch.optim.AdamW([{"params": adam_decay, "weight_decay": weight_decay},
                              {"params": adam_no_decay, "weight_decay": 0.0}], lr=lr, betas=(0.9, 0.95), eps=1e-8)
    return ([Muon(muon_params, lr=lr, weight_decay=weight_decay)] if muon_params else []) + [adam]


def make_grad_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def lr_factor(step, progress, warmup_steps, cooldown_frac, floor):
    """Warmup-stable-decay on *time*: linear warmup (steps), constant, then a
    1-sqrt cooldown over the last `cooldown_frac` of the training time."""
    w = min(1.0, (step + 1) / max(1, warmup_steps))
    start = 1.0 - cooldown_frac
    if progress < start:
        return w
    r = min(1.0, (progress - start) / max(1e-9, cooldown_frac))
    return w * (floor + (1.0 - floor) * (1.0 - math.sqrt(r)))


def loss_from(out, z_loss_coef):
    s = out.sum(0)
    n = s[1].clamp(min=1)
    return s[0] / n, (s[0] + z_loss_coef * s[2]) / n


# ---------------------------------------------------------------------------
# Data: stream of 1024-token windows (round-robin over the cleaned files,
# documents separated by <eos>), byte positions saved for resume. Each GPU
# (rank) starts at a different offset of every file.
# ---------------------------------------------------------------------------
class WindowStream:
    def __init__(self, file_paths, window, rank=0, world=1, start_positions=None):
        self.files = [Path(p) for p in file_paths if Path(p).stat().st_size > 0]
        self.window = window
        self.positions, self.unaligned = {}, set()
        for p in self.files:
            if start_positions and p.name in start_positions:
                self.positions[p.name] = int(start_positions[p.name])
            else:
                self.positions[p.name] = (p.stat().st_size * rank) // world
                if self.positions[p.name] > 0:
                    self.unaligned.add(p.name)

    def __iter__(self):
        buffer = []
        readers = [self._read_lines(p) for p in self.files]
        for reader in itertools.cycle(readers):
            ids = encode(next(reader))
            if not ids:
                continue
            buffer.extend(ids)
            buffer.append(EOS_ID)
            while len(buffer) >= self.window:
                yield buffer[:self.window]
                buffer = buffer[self.window:]

    def _read_lines(self, path):
        name, size = path.name, path.stat().st_size
        while True:
            start = self.positions[name] if self.positions[name] < size else 0
            with open(path, "rb") as f:
                f.seek(start)
                pos = start
                if name in self.unaligned:                  # started mid-line: skip to the next line
                    pos += len(f.readline())
                    self.unaligned.discard(name)
                while True:
                    raw = f.readline()
                    if not raw:
                        break
                    pos += len(raw)
                    self.positions[name] = pos
                    line = raw.decode("utf-8", errors="ignore").strip()
                    if line:
                        yield line
            self.positions[name] = 0


def make_example_batch(windows, P, Q, device):
    """windows: list of 1024-token lists -> encoder ids [B,P], decoder input
    [B,Q] (start token + shifted continuation), targets [B,Q]."""
    w = torch.tensor(windows, dtype=torch.long)
    enc = w[:, :P]
    tgt = w[:, P:P + Q]
    dec_in = torch.cat([torch.full((w.shape[0], 1), DEC_START_ID, dtype=torch.long), tgt[:, :-1]], dim=1)
    return enc.to(device, non_blocking=True), dec_in.to(device, non_blocking=True), tgt.to(device, non_blocking=True)


def choose_shape(shapes, seed, step, micro):
    """Same (prompt, continuation) split on every GPU for a given micro-step."""
    rnd = random.Random(seed * 1_000_003 + step * 97 + micro)
    return rnd.choices([(s[0], s[1]) for s in shapes], weights=[s[2] for s in shapes])[0]


# ---------------------------------------------------------------------------
# Checkpoints and Kaggle push
# ---------------------------------------------------------------------------
def step_of(path):
    return int(Path(path).stem.split("_step")[1].split("_tokens")[0])


def find_latest_checkpoint(prefix, search_dirs, extract_to):
    cands = []
    for d in search_dirs:
        d = Path(d)
        if d.exists():
            cands += [p for p in d.rglob(f"{prefix}_step*.pt")]
    best_zip = None
    for d in search_dirs:
        d = Path(d)
        if not d.exists():
            continue
        for zpath in d.rglob("*.zip"):
            try:
                with zipfile.ZipFile(zpath) as zf:
                    for name in zf.namelist():
                        base = Path(name).name
                        if base.startswith(f"{prefix}_step") and base.endswith(".pt"):
                            if best_zip is None or step_of(base) > best_zip[0]:
                                best_zip = (step_of(base), zpath, name)
            except zipfile.BadZipFile:
                continue
    best_file = max(cands, key=step_of) if cands else None
    if best_zip is not None and (best_file is None or best_zip[0] > step_of(best_file)):
        _, zpath, name = best_zip
        dest = Path(extract_to) / Path(name).name
        with zipfile.ZipFile(zpath) as zf, zf.open(name) as src, open(dest, "wb") as dst:
            shutil.copyfileobj(src, dst)
        return dest
    return best_file


def prune_old_checkpoints(ckpt_dir, prefix, keep=2):
    ckpts = sorted(Path(ckpt_dir).glob(f"{prefix}_step*.pt"), key=step_of)
    for p in ckpts[:-keep] if len(ckpts) > keep else []:
        p.unlink()


def push_checkpoint_dataset(output_dir, dataset_id, message):
    kdir = Path.home() / ".kaggle"
    if not ((kdir / "access_token").exists() or (kdir / "kaggle.json").exists() or os.environ.get("KAGGLE_API_TOKEN")):
        print("Skipping checkpoint push: no Kaggle API token.", flush=True)
        return False
    slug = dataset_id.split("/")[1]
    with open(Path(output_dir) / "dataset-metadata.json", "w") as f:
        json.dump({"title": slug, "id": dataset_id, "licenses": [{"name": "CC0-1.0"}]}, f)
    try:
        exists = subprocess.run(["kaggle", "datasets", "status", dataset_id], capture_output=True, text=True,
                                timeout=300).returncode == 0
        cmd = (["kaggle", "datasets", "version", "-p", str(output_dir), "-m", message, "-r", "zip"] if exists
               else ["kaggle", "datasets", "create", "-p", str(output_dir), "-r", "zip"])
        t0 = time.time()
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)
    except (OSError, subprocess.TimeoutExpired) as exc:
        print(f"Checkpoint push failed: {exc}", flush=True)
        return False
    if result.returncode != 0:
        print(f"Checkpoint push failed: {(result.stderr or result.stdout).strip()[-500:]}", flush=True)
        return False
    print(f"Pushed checkpoint dataset in {time.time() - t0:.0f}s ({message}).", flush=True)
    return True


def load_for_inference(path, device, half=False):
    """fp32 weights + fp16 autocast on GPU (same numerics as training)."""
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    model = build_model(ckpt["config"])
    model.load_state_dict(ckpt["model_state"])
    meta = {k: ckpt.get(k) for k in ("step", "tokens_seen", "config", "run_id")}
    del ckpt
    model = model.to(device).eval()
    if half and device.type == "cuda":
        model = model.half()
        model.rope_cos, model.rope_sin = model.rope_cos.float(), model.rope_sin.float()
    return model, meta


# ---------------------------------------------------------------------------
# Trainer (one process per GPU)
# ---------------------------------------------------------------------------
def _probe_micro_batch(model, cfg, device, P, Q):
    """Largest micro-batch (1024-token examples) that fits the memory budget
    on this GPU, with the real optimizers; weights are restored afterwards."""
    saved = {k: v.detach().clone().cpu() for k, v in model.state_dict().items()}
    results = {}
    for bs in cfg["candidate_micro_bs"]:
        optims = build_optimizers(model, cfg["optimizer"], 1e-5, 0.0)
        scaler = make_grad_scaler(True)
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)
        try:
            windows = [[random.randrange(4, 1000) for _ in range(P + Q)] for _ in range(bs)]
            enc, dec_in, tgt = make_example_batch(windows, P, Q, device)
            times = []
            for _ in range(3):
                torch.cuda.synchronize(device)
                t0 = time.time()
                for o in optims:
                    o.zero_grad(set_to_none=True)
                _, total = loss_from(model(enc, dec_in, targets=tgt), 1e-4)
                scaler.scale(total).backward()
                for o in optims:
                    scaler.step(o)
                scaler.update()
                torch.cuda.synchronize(device)
                times.append(time.time() - t0)
            peak = torch.cuda.max_memory_allocated(device) / 1e9
            ok = peak <= cfg["memory_budget_gb"]
            results[bs] = (peak, min(times[1:]), ok)
            if not ok:
                break
        except Exception as exc:                           # noqa: BLE001
            if "out of memory" not in str(exc).lower():
                raise
            results[bs] = (None, None, False)
            break
        finally:
            for o in optims:
                o.zero_grad(set_to_none=True)
            del optims
            model.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
    model.load_state_dict(saved)
    return results


def train_main(config_path):
    cfg = json.load(open(config_path))
    distributed = int(os.environ.get("WORLD_SIZE", "1")) > 1
    cuda = torch.cuda.is_available()
    if distributed:
        dist.init_process_group("nccl" if cuda else "gloo", timeout=datetime.timedelta(minutes=90))
        rank, world, local_rank = dist.get_rank(), dist.get_world_size(), int(os.environ.get("LOCAL_RANK", 0))
    else:
        rank, world, local_rank = 0, 1, 0
    device = torch.device(f"cuda:{local_rank}") if cuda else torch.device("cpu")
    if cuda:
        torch.cuda.set_device(device)
        torch.backends.cuda.matmul.allow_tf32 = True
    main = rank == 0

    def log(*a):
        if main:
            print(*a, flush=True)

    load_tokenizer(cfg["tokenizer_path"])
    model_cfg = cfg["model"]
    prefix = cfg["ckpt_prefix"]
    ckpt_dir = Path(cfg["checkpoint_dir"])
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    session_start, session_deadline = cfg["session_start_epoch"], cfg["session_start_epoch"] + 60 * cfg["time_budget_min"]
    total_seconds = 3600 * cfg["total_train_hours"]
    log(f"[D] world={world} device={device} | budget this session {cfg['time_budget_min']} min, "
        f"whole run {cfg['total_train_hours']} h | optimizer={cfg['optimizer']}")

    torch.manual_seed(cfg["seed"])
    random.seed(cfg["seed"] + rank)
    model = build_model(model_cfg).to(device)
    n_params = count_params(model)
    log(f"[D] parameters: {n_params / 1e6:.1f}M")

    # ---- resume ----------------------------------------------------------
    step, tokens_seen, seconds_before, rank_positions, ckpt = 0, 0, 0.0, None, None
    phase = cfg.get("phase_id", "main")        # a new phase id = continue training with a fresh LR schedule
    phase_start_step = 0
    latest = find_latest_checkpoint(prefix, [ckpt_dir] + cfg.get("resume_search_dirs", []), ckpt_dir) if main else None
    if distributed:
        box = [str(latest) if latest else ""]
        dist.broadcast_object_list(box, src=0)
        latest = Path(box[0]) if box[0] else None
    if latest is not None:
        ckpt = torch.load(latest, map_location="cpu", weights_only=False)
        assert ckpt["config"] == model_cfg, "Checkpoint config does not match MODEL_D_CONFIG"
        model.load_state_dict(ckpt["model_state"])
        step, tokens_seen = ckpt["step"], ckpt["tokens_seen"]
        same_session = ckpt.get("session_start_epoch") == session_start
        rank_positions = ckpt.get("data_state", {}).get("rank_positions")
        ckpt_phase = ckpt.get("phase_id", "main")
        if ckpt_phase == phase:
            seconds_before = ckpt["seconds_before_session"] if same_session else ckpt["progress_seconds"]
            phase_start_step = ckpt.get("phase_start_step", 0)
        else:
            seconds_before, phase_start_step = 0.0, step
            log(f"[D] new training phase '{phase}' (previous: '{ckpt_phase}') -> fresh LR schedule from step {step:,}")
        log(f"[D] resumed from {latest.name}: step={step:,} tokens={tokens_seen:,} "
            f"phase progress={seconds_before / total_seconds:.1%} ({'same' if same_session else 'new'} session)")
    else:
        log("[D] no checkpoint found -- starting fresh")

    # ---- micro-batch size ---------------------------------------------------
    shapes = cfg["shapes"]
    Pp, Qp = max(((s[0], s[1]) for s in shapes), key=lambda s: s[1])      # longest decoder = most memory
    if cfg.get("fixed_micro_bs") or not cuda:
        micro_bs, sec_micro = cfg.get("fixed_micro_bs") or 2, None
    else:
        res = _probe_micro_batch(model, cfg, device, Pp, Qp)
        fits = [bs for bs, (_, _, ok) in res.items() if ok]
        if not fits:
            model.grad_ckpt = True
            log("[D] nothing fit without activation checkpointing -> turning it on")
            res = _probe_micro_batch(model, cfg, device, Pp, Qp)
            fits = [bs for bs, (_, _, ok) in res.items() if ok]
        for bs, (peak, sec, ok) in res.items():
            log(f"  micro-batch {bs:3d} x 1024 -> " + (f"{peak:5.2f} GB, {sec:.2f}s, {bs * 1024 / sec:,.0f} tok/s/GPU"
                                                      if peak else "OOM") + ("" if ok else "  [no]"))
        best = max(fits) if fits else 0
        if distributed:
            t = torch.tensor([best], device=device)
            dist.all_reduce(t, op=dist.ReduceOp.MIN)
            best = int(t.item())
        assert best > 0, "No micro-batch fits in memory"
        micro_bs, sec_micro = best, res[best][1]
    accum = max(1, round(cfg["tokens_per_step_target"] / (micro_bs * 1024 * world)))
    tokens_per_step = micro_bs * 1024 * world * accum
    log(f"[D] micro-batch {micro_bs} x 1024 tokens x {world} GPU(s) x accum {accum} = {tokens_per_step:,} tokens/step"
        + (f" | probe ~{micro_bs * 1024 * world / sec_micro:,.0f} tok/s" if sec_micro else ""))

    optims = build_optimizers(model, cfg["optimizer"], cfg["peak_lr"], cfg["weight_decay"])
    scaler = make_grad_scaler(cuda)
    if ckpt is not None:
        if ckpt.get("optimizer_kind") == cfg["optimizer"]:
            for o, s in zip(optims, ckpt["optim_states"]):
                o.load_state_dict(s)
            scaler.load_state_dict(ckpt["scaler_state"])
        else:
            log("[D] optimizer changed since the checkpoint -> fresh optimizer state")
        model.grad_ckpt = model.grad_ckpt or ckpt.get("grad_ckpt", False)
        del ckpt
    train_model = model
    if distributed:
        from torch.nn.parallel import DistributedDataParallel as DDP
        train_model = DDP(model, device_ids=[local_rank] if cuda else None, gradient_as_bucket_view=True)

    positions = rank_positions[rank] if rank_positions and rank < len(rank_positions) else None
    files = sorted(Path(cfg["data_dir"]).glob("clean_*.txt"))
    assert files, "No clean_*.txt files in the data folder"
    stream = WindowStream(files, 1024, rank, world, positions)
    data_iter = iter(stream)
    log(f"[D] training files: {[p.name for p in files]} | shapes (prompt, continuation, weight): {shapes}")

    def save_checkpoint(reason):
        gathered = [None] * world
        if distributed:
            dist.all_gather_object(gathered, dict(stream.positions))
        else:
            gathered = [dict(stream.positions)]
        path = None
        if main:
            now = time.time()
            state = {
                "model_state": model.state_dict(), "optim_states": [o.state_dict() for o in optims],
                "optimizer_kind": cfg["optimizer"], "scaler_state": scaler.state_dict(), "step": step,
                "tokens_seen": tokens_seen, "config": model_cfg, "arch_id": ARCH_ID, "run_id": cfg["run_id"],
                "session_start_epoch": session_start, "seconds_before_session": seconds_before,
                "progress_seconds": seconds_before + (now - session_start), "grad_ckpt": model.grad_ckpt,
                "phase_id": phase, "phase_start_step": phase_start_step,
                "data_state": {"rank_positions": gathered, "world": world},
                "train_cfg": {k: cfg[k] for k in ("peak_lr", "shapes", "optimizer", "total_train_hours")},
                "reason": reason,
            }
            path = ckpt_dir / f"{prefix}_step{step}_tokens{tokens_seen}.pt"
            tmp = path.with_name(path.name + ".tmp")
            torch.save(state, tmp)
            tmp.replace(path)
            prune_old_checkpoints(ckpt_dir, prefix, keep=2)
        if distributed:
            dist.barrier()
        return path

    # ---- loop --------------------------------------------------------------
    train_model.train()
    t_loop, last_ckpt, last_log_t, last_log_tokens = time.time(), time.time(), time.time(), tokens_seen
    start_tokens = tokens_seen
    bad_streak, skipped, stop_reason = 0, 0, "unknown"
    ce_log, n_log = torch.zeros((), device=device), 0
    flags = torch.zeros(3, device=device)
    while True:
        if main:
            now = time.time()
            progress = (seconds_before + (now - session_start)) / total_seconds
            stop = 1.0 if (progress >= 1.0 or now >= session_deadline) else 0.0
            do_ckpt = 1.0 if (now - last_ckpt) / 60 >= cfg["checkpoint_every_min"] else 0.0
            flags = torch.tensor([progress, stop, do_ckpt], device=device)
        if distributed:
            dist.broadcast(flags, src=0)
        progress, stop, do_ckpt = float(flags[0]), bool(flags[1] > 0.5), bool(flags[2] > 0.5)
        if stop:
            stop_reason = "schedule complete" if progress >= 1.0 else "session time budget reached"
            break
        lr = cfg["peak_lr"] * lr_factor(step - phase_start_step, progress, cfg["warmup_steps"], cfg["cooldown_frac"], cfg["min_lr_frac"])
        for o in optims:
            for g in o.param_groups:
                g["lr"] = lr
            o.zero_grad(set_to_none=True)
        for micro in range(accum):
            P, Q = choose_shape(shapes, cfg["seed"], step, micro)
            enc, dec_in, tgt = make_example_batch([next(data_iter) for _ in range(micro_bs)], P, Q, device)
            sync_ctx = train_model.no_sync() if (distributed and micro < accum - 1) else contextlib.nullcontext()
            with sync_ctx:
                ce, total = loss_from(train_model(enc, dec_in, targets=tgt), cfg["z_loss_coef"])
                scaler.scale(total / accum).backward()
            ce_log += ce.detach() / accum
            tokens_seen += micro_bs * 1024 * world
        for o in optims:
            scaler.unscale_(o)
        gnorm = float(torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"]))
        for o in optims:
            scaler.step(o)
        scaler.update()
        n_log += 1
        step += 1
        if math.isfinite(gnorm):
            bad_streak = 0
        else:
            bad_streak += 1
            skipped += 1
            if bad_streak >= 50:
                stop_reason = "50 non-finite steps in a row"
                break
        if distributed and step % cfg["sync_every"] == 0:
            for p in model.parameters():                     # remove any tiny drift between GPUs
                dist.broadcast(p.data, src=0)
        if step % cfg["log_every"] == 0 or step == 1:
            if distributed:
                dist.all_reduce(ce_log)
                ce_log /= world
            now = time.time()
            tps = (tokens_seen - last_log_tokens) / max(1e-6, now - last_log_t)
            log(f"  step {step:6d}  loss={ce_log.item() / max(1, n_log):.4f}  lr={lr:.2e}  gnorm={gnorm:.2f}  "
                f"tokens={tokens_seen:,}  tok/s={tps:,.0f}  elapsed={(now - t_loop) / 60:.1f} min  "
                f"progress={progress:.1%}  [{datetime.datetime.now():%H:%M:%S}]")
            ce_log.zero_()
            n_log, last_log_t, last_log_tokens = 0, now, tokens_seen
        if do_ckpt:
            path = save_checkpoint("periodic")
            last_ckpt = time.time()
            if main:
                print(f"  [checkpoint saved -> {path.name}]", flush=True)
                if cfg["push"]:
                    push_checkpoint_dataset(cfg["output_dir"], cfg["dataset_id"],
                                            f"{cfg['run_id']}: step={step}, tokens={tokens_seen:,} (in progress)")
            if distributed:
                dist.barrier()
    path = save_checkpoint(stop_reason)
    loop_min = (time.time() - t_loop) / 60
    summary = {"stop_reason": stop_reason, "step": step, "tokens_seen": tokens_seen,
               "session_tokens": tokens_seen - start_tokens, "loop_minutes": round(loop_min, 1),
               "tokens_per_sec": round((tokens_seen - start_tokens) / max(1e-6, loop_min * 60), 1),
               "world": world, "micro_bs": micro_bs, "accum": accum, "tokens_per_step": tokens_per_step,
               "skipped_steps": skipped, "grad_ckpt": model.grad_ckpt, "n_params": n_params, "phase_id": phase,
               "phase_start_step": phase_start_step,
               "final_checkpoint": path.name if path else None, "progress": round(progress, 4),
               "peak_mem_gb": round(torch.cuda.max_memory_allocated(device) / 1e9, 2) if cuda else 0.0}
    if main:
        json.dump(summary, open(Path(cfg["output_dir"]) / "train_summary.json", "w"), indent=1)
        log(f"[D] stopped ({stop_reason}) at step={step:,}, tokens={tokens_seen:,} after {loop_min:.1f} min "
            f"-> {path.name}")
    if distributed:
        dist.barrier()
        dist.destroy_process_group()
    if stop_reason.startswith("50 non-finite"):
        sys.exit(3)


if __name__ == "__main__":
    train_main(sys.argv[1])

In [ ]:
%%writefile /kaggle/working/c_model.py
"""c_model.py -- Model C (prefix-LM) classes, copied from p1_07_model_C.ipynb."""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint

decode = None          # only used by C's text generation, not needed here


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        return (x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight.float()).to(dtype)


def build_rope_cache(seq_len, d_head, theta=10000.0):
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_head, 2).float() / d_head))
    freqs = torch.outer(torch.arange(seq_len).float(), inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    """x [B,H,T,d]; cos/sin [B or 1, T, d/2]. Rotation in fp32."""
    x1, x2 = x[..., 0::2].float(), x[..., 1::2].float()
    cos, sin = cos[:, None].float(), sin[:, None].float()
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2).to(x.dtype)


def prefix_lm_mask(prefix_len, T, device):
    """Bool [B, 1, T, T]: key k visible to query q iff k <= q or k < prefix_len[b]."""
    idx = torch.arange(T, device=device)
    causal = idx[None, :, None] >= idx[None, None, :]
    in_prefix = idx[None, None, :] < prefix_len[:, None, None]
    return (causal | in_prefix)[:, None]


class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, h, kv, dh, b = cfg["d_model"], cfg["n_head"], cfg["n_kv_head"], cfg["d_head"], cfg["bias"]
        self.n_head, self.n_kv_head, self.d_head = h, kv, dh
        self.q_proj = nn.Linear(d, h * dh, bias=b)
        self.k_proj = nn.Linear(d, kv * dh, bias=b)
        self.v_proj = nn.Linear(d, kv * dh, bias=b)
        self.o_proj = nn.Linear(h * dh, d, bias=b)
        self.q_norm = RMSNorm(dh) if cfg["qk_norm"] else None
        self.k_norm = RMSNorm(dh) if cfg["qk_norm"] else None

    def forward(self, x, cos, sin, attn_mask, cache=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        if self.q_norm is not None:
            q, k = self.q_norm(q), self.k_norm(k)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if cache is not None:
            k, v = cache.update(k, v)
        n_rep = self.n_head // self.n_kv_head
        if n_rep > 1:
            k, v = k.repeat_interleave(n_rep, dim=1), v.repeat_interleave(n_rep, dim=1)
        if attn_mask is None:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=T > 1)
        else:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head))


class GeluMLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.up_proj = nn.Linear(cfg["d_model"], cfg["d_ff"], bias=cfg["bias"])
        self.down_proj = nn.Linear(cfg["d_ff"], cfg["d_model"], bias=cfg["bias"])

    def forward(self, x):
        return self.down_proj(F.gelu(self.up_proj(x), approximate="tanh"))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg["d_model"])
        self.attn = Attention(cfg)
        self.mlp_norm = RMSNorm(cfg["d_model"])
        self.mlp = GeluMLP(cfg)

    def _inner(self, x, cos, sin, attn_mask, cache):
        x = x + self.attn(self.attn_norm(x), cos, sin, attn_mask, cache)
        return x + self.mlp(self.mlp_norm(x))

    def forward(self, x, cos, sin, attn_mask, cache=None, use_checkpoint=False):
        if use_checkpoint and cache is None and self.training:
            return grad_checkpoint(self._inner, x, cos, sin, attn_mask, None, use_reentrant=False)
        return self._inner(x, cos, sin, attn_mask, cache)


class KVCache:
    def __init__(self, batch_size, n_kv_head, max_len, d_head, device, dtype):
        self.k = torch.zeros(batch_size, n_kv_head, max_len, d_head, device=device, dtype=dtype)
        self.v = torch.zeros_like(self.k)
        self.pos = 0

    def update(self, k, v):
        T = k.shape[2]
        self.k[:, :, self.pos:self.pos + T] = k
        self.v[:, :, self.pos:self.pos + T] = v
        self.pos += T
        return self.k[:, :, :self.pos], self.v[:, :, :self.pos]


class PrefixLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed = nn.Embedding(config["vocab_size"], config["d_model"])
        self.blocks = nn.ModuleList([Block(config) for _ in range(config["n_layer"])])
        self.norm_f = RMSNorm(config["d_model"])
        self.head = nn.Linear(config["d_model"], config["vocab_size"], bias=False)
        self.head.weight = self.embed.weight
        cos, sin = build_rope_cache(config["max_seq_len"], config["d_head"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config["n_layer"]))

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

    def encode(self, input_ids, attention_mask=None, mode="prefix", prefix_len=None):
        """Contract method: hidden states. mode 'causal' (prefix 0), 'prefix'
        (needs prefix_len [B]) or 'bidirectional' (whole input is prefix)."""
        B, T = input_ids.shape
        if mode == "causal" or prefix_len is None and mode == "prefix":
            prefix_len = torch.zeros(B, dtype=torch.long, device=input_ids.device)
        elif mode == "bidirectional":
            prefix_len = torch.full((B,), T, dtype=torch.long, device=input_ids.device)
        mask = prefix_lm_mask(prefix_len, T, input_ids.device) if bool((prefix_len > 0).any()) else None
        x = self.embed(input_ids)
        cos, sin = self.rope_cos[None, :T], self.rope_sin[None, :T]
        use_ckpt = self.training and USE_GRAD_CHECKPOINTING
        for blk in self.blocks:
            x = blk(x, cos, sin, mask, use_checkpoint=use_ckpt)
        return self.norm_f(x)

    def forward(self, ids, prefix_len=None, targets=None):
        """No targets -> logits. With targets -> [1, 3] = [loss sum, number of
        loss tokens, sum of logsumexp^2 (z-loss)] over positions that predict
        continuation tokens (t >= prefix_len - 1). Sums (not means) so
        DataParallel replicas combine exactly."""
        use_amp = self.training and ids.is_cuda
        if prefix_len is None:
            prefix_len = torch.zeros(ids.shape[0], dtype=torch.long, device=ids.device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = self.head(self.encode(ids, mode="prefix", prefix_len=prefix_len))
        if targets is None:
            return logits
        logits32 = logits.float()
        lse = torch.logsumexp(logits32, dim=-1)
        nll = lse - logits32.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
        pos = torch.arange(ids.shape[1], device=ids.device)[None, :]
        w = (pos >= (prefix_len[:, None] - 1)).float()
        return torch.stack([(nll * w).sum(), w.sum(), (lse.pow(2) * w).sum()]).unsqueeze(0)

    def step(self, ids, positions, caches, attn_mask):
        """Generation step with a KV cache. ids/positions [B, T]; attn_mask
        bool [B, 1, T, cache_len + T]."""
        x = self.embed(ids)
        cos, sin = self.rope_cos[positions], self.rope_sin[positions]
        for blk, cache in zip(self.blocks, caches):
            x = blk(x, cos, sin, attn_mask, cache=cache)
        return self.head(self.norm_f(x))

    def generate(self, input_ids, max_new_tokens, kv_cache=True, sampler="greedy", **kw):
        """Contract method (greedy, one prompt): the prompt is the prefix."""
        assert kv_cache and sampler == "greedy"
        toks = generate_batch(self, [input_ids[0].tolist()], max_new_tokens=max_new_tokens, **kw)[0]
        return torch.tensor([input_ids[0].tolist() + toks], device=input_ids.device)


@torch.no_grad()
def generate_batch(model, prompt_id_lists, max_new_tokens=288, max_context=None, rep_penalty=1.0,
                   no_repeat_ngram=0, target_words=None, banned=(), vocab_limit=None):
    """Greedy continuations for several prompts at once (left-padded). The
    prompt is the bidirectional prefix; continuation is causal. Optional:
    repetition penalty over generated tokens, n-gram blocking inside the
    continuation, stop once every row has > target_words words."""
    model.eval()
    device = next(model.parameters()).device
    dtype = next(model.parameters()).dtype
    cfg = model.config
    max_context = max_context or cfg["max_seq_len"]
    prompts = [p[-(max_context - max_new_tokens):] for p in prompt_id_lists]
    B, T0 = len(prompts), max(len(p) for p in prompts)
    pad = torch.tensor([T0 - len(p) for p in prompts], device=device)
    ids = torch.zeros(B, T0, dtype=torch.long, device=device)
    for b, p in enumerate(prompts):
        ids[b, T0 - len(p):] = torch.tensor(p, device=device)
    total = T0 + max_new_tokens
    caches = [KVCache(B, cfg["n_kv_head"], total, cfg["d_head"], device, dtype) for _ in range(cfg["n_layer"])]
    idx = torch.arange(total, device=device)
    key_ok = idx[None, :] >= pad[:, None]
    positions = (idx[None, :] - pad[:, None]).clamp(min=0)
    # Prefill: the prompt attends to itself in both directions (no padding keys);
    # every query may see itself so padded rows are never fully masked.
    mask = (key_ok[:, None, :T0] | torch.eye(T0, dtype=torch.bool, device=device)[None])[:, None]
    logits = model.step(ids, positions[:, :T0], caches, mask)[:, -1, :].float()
    generated = [[] for _ in range(B)]
    seen = [dict() for _ in range(B)]
    for t in range(max_new_tokens):
        if banned:
            logits[:, list(banned)] = -float("inf")
        if vocab_limit is not None:
            logits[:, vocab_limit:] = -float("inf")     # ids the tokenizer cannot decode
        if rep_penalty != 1.0 and t > 0:
            prev = torch.tensor(generated, device=device)
            sc = logits.gather(1, prev)
            logits.scatter_(1, prev, torch.where(sc > 0, sc / rep_penalty, sc * rep_penalty))
        if no_repeat_ngram and t >= no_repeat_ngram - 1:
            n = no_repeat_ngram
            for b in range(B):
                bad = seen[b].get(tuple(generated[b][-(n - 1):]) if n > 1 else ())
                if bad:
                    logits[b, list(bad)] = -float("inf")
        nxt = logits.argmax(dim=-1)
        for b, tok in enumerate(nxt.tolist()):
            generated[b].append(tok)
            if no_repeat_ngram and len(generated[b]) >= no_repeat_ngram:
                g = generated[b][-no_repeat_ngram:]
                seen[b].setdefault(tuple(g[:-1]), set()).add(g[-1])
        if t == max_new_tokens - 1:
            break
        if target_words and t >= 120 and t % 16 == 0 and all(
                len(decode(g).split()) > target_words for g in generated):
            break
        cur = T0 + t
        logits = model.step(nxt[:, None], positions[:, cur:cur + 1], caches,
                            key_ok[:, None, None, :cur + 1])[:, -1, :].float()
    return generated


def build_model(config):
    """Contract function."""
    return PrefixLM(config)


USE_GRAD_CHECKPOINTING = True

In [ ]:
%%writefile /kaggle/working/a_model.py
"""a_model.py -- Model A (Llama-style decoder) classes, copied from p1_03_model_A.ipynb."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint
class RMSNorm(nn.Module):
    """RMSNorm instead of LayerNorm: one learned scale, no learned bias,
    no mean-subtraction -- cheaper, and the modern-architecture standard
    this whole design follows (Llama, Mistral, etc.).
    """
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        norm = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (norm * self.weight.float()).to(dtype)


def build_rope_cache(seq_len, d_head, theta=10000.0, device="cpu"):
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_head, 2, device=device).float() / d_head))
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.outer(t, inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    x1, x2 = x[..., 0::2], x[..., 1::2]
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    rotated = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return rotated.flatten(-2)


try:
    _probe = torch.randn(1, 1, 2, 4)
    F.scaled_dot_product_attention(_probe, _probe, _probe, enable_gqa=True)
    _SDPA_HAS_ENABLE_GQA = True
except TypeError:
    _SDPA_HAS_ENABLE_GQA = False


class GQAAttention(nn.Module):
    """Grouped-query attention: fewer KV heads than query heads, which
    shrinks the KV cache proportionally (here, 7x) at a small quality cost.
    """
    def __init__(self, d_model, n_head, n_kv_head, d_head):
        super().__init__()
        self.n_head, self.n_kv_head, self.d_head = n_head, n_kv_head, d_head
        self.q_proj = nn.Linear(d_model, n_head * d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_head * d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_head * d_head, bias=False)
        self.o_proj = nn.Linear(n_head * d_head, d_model, bias=False)

    def forward(self, x, cos, sin, cache=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)

        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if cache is not None:
            k, v = cache.update(k, v)

        is_causal = q.shape[2] > 1

        if _SDPA_HAS_ENABLE_GQA:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=is_causal, enable_gqa=True)
        else:
            n_rep = self.n_head // self.n_kv_head
            k_rep = k.repeat_interleave(n_rep, dim=1)
            v_rep = v.repeat_interleave(n_rep, dim=1)
            out = F.scaled_dot_product_attention(q, k_rep, v_rep, is_causal=is_causal)

        out = out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head)
        return self.o_proj(out)


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class Block(nn.Module):
    """One transformer layer: pre-norm attention, pre-norm MLP, both as
    residual additions.
    """
    def __init__(self, d_model, n_head, n_kv_head, d_head, d_ff):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = GQAAttention(d_model, n_head, n_kv_head, d_head)
        self.mlp_norm = RMSNorm(d_model)
        self.mlp = SwiGLU(d_model, d_ff)

    def _forward_inner(self, x, cos, sin, cache):
        x = x + self.attn(self.attn_norm(x), cos, sin, cache)
        x = x + self.mlp(self.mlp_norm(x))
        return x

    def forward(self, x, cos, sin, cache=None, use_checkpoint=False):
        # Checkpointing needs a cache-free forward (no in-place cache
        # mutation across a recomputed backward pass) and only makes
        # sense during training, never during KV-cache generation.
        if use_checkpoint and cache is None and self.training:
            return grad_checkpoint(self._forward_inner, x, cos, sin, None, use_reentrant=False)
        return self._forward_inner(x, cos, sin, cache)


class KVCache:
    """A pre-allocated, fixed-size cache for one layer's keys and values."""
    def __init__(self, batch_size, n_kv_head, max_seq_len, d_head, device, dtype):
        shape = (batch_size, n_kv_head, max_seq_len, d_head)
        self.k = torch.zeros(shape, device=device, dtype=dtype)
        self.v = torch.zeros(shape, device=device, dtype=dtype)
        self.pos = 0

    def update(self, k, v):
        T = k.shape[2]
        self.k[:, :, self.pos:self.pos + T] = k
        self.v[:, :, self.pos:self.pos + T] = v
        self.pos += T
        return self.k[:, :, :self.pos], self.v[:, :, :self.pos]


class MiniGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed = nn.Embedding(config["vocab_size"], config["d_model"])
        self.blocks = nn.ModuleList([
            Block(config["d_model"], config["n_head"], config["n_kv_head"],
                  config["d_head"], config["d_ff"])
            for _ in range(config["n_layer"])
        ])
        self.norm_f = RMSNorm(config["d_model"])
        self.head = nn.Linear(config["d_model"], config["vocab_size"], bias=False)
        self.head.weight = self.embed.weight  # weight tying

        cos, sin = build_rope_cache(config["max_seq_len"], config["d_head"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config["n_layer"]))

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def new_kv_caches(self, batch_size, device, dtype):
        return [
            KVCache(batch_size, self.config["n_kv_head"], self.config["max_seq_len"],
                    self.config["d_head"], device, dtype)
            for _ in range(self.config["n_layer"])
        ]

    def encode(self, input_ids, attention_mask=None, mode="causal"):
        """Contract method: raw hidden states, no LM head. attention_mask
        is accepted for interface compatibility with later phases (e.g. a
        padded batch) but unused here -- Phase 1 pretraining always packs
        full, unpadded windows, so every position is valid.
        """
        assert mode == "causal", "MiniGPT is decoder-only; only causal encoding is supported."
        B, T = input_ids.shape
        x = self.embed(input_ids)
        cos = self.rope_cos[:T].to(x.device)
        sin = self.rope_sin[:T].to(x.device)
        use_ckpt = self.training and USE_GRAD_CHECKPOINTING
        for blk in self.blocks:
            x = blk(x, cos, sin, cache=None, use_checkpoint=use_ckpt)
        return self.norm_f(x)

    def forward(self, ids, caches=None):
        if caches is None:
            return self.head(self.encode(ids))
        # Generation path (KV cache present): identical to encode() but
        # threading a per-layer cache through and offsetting RoPE by how
        # much of the cache is already filled.
        B, T = ids.shape
        x = self.embed(ids)
        pos_offset = caches[0].pos
        cos = self.rope_cos[pos_offset:pos_offset + T].to(x.device)
        sin = self.rope_sin[pos_offset:pos_offset + T].to(x.device)
        for i, blk in enumerate(self.blocks):
            x = blk(x, cos, sin, cache=caches[i], use_checkpoint=False)
        x = self.norm_f(x)
        return self.head(x)

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, kv_cache=True, sampler="greedy", top_p=0.9, temperature=1.0):
        """Contract method. `sampler`: "greedy" (argmax, deterministic --
        what the smoke-test submission uses) or "top_p" (nucleus sampling).
        Stops early (per sequence, via a done mask) at <eos> or max_seq_len.
        """
        assert kv_cache, "This notebook's generate() only implements the KV-cache path -- it's what training will actually ship."
        self.eval()
        device, dtype = input_ids.device, next(self.parameters()).dtype

        # A prompt longer than the model's context has nowhere to put
        # max_new_tokens (or, if it's longer than max_seq_len outright,
        # RoPE's precomputed cos/sin tables don't even cover it -- a hard
        # crash, not a quality issue). Truncate to the most recent tokens
        # -- the ones closest to where generation continues -- rather
        # than letting this crash the whole generation loop over one long
        # dev-set example.
        max_prompt_len = self.config["max_seq_len"] - max_new_tokens
        if input_ids.shape[1] > max_prompt_len:
            print(f"  [generate() warning: prompt has {input_ids.shape[1]} tokens, "
                  f"truncating to the last {max_prompt_len} to leave room for "
                  f"{max_new_tokens} new tokens within max_seq_len={self.config['max_seq_len']}]")
            input_ids = input_ids[:, -max_prompt_len:]

        B, T0 = input_ids.shape
        max_total = min(T0 + max_new_tokens, self.config["max_seq_len"])

        caches = self.new_kv_caches(batch_size=B, device=device, dtype=dtype)
        generated = input_ids
        done = torch.zeros(B, dtype=torch.bool, device=device)

        logits = self.forward(input_ids, caches=caches)  # prefill: process the whole prompt at once
        next_ids = self._sample(logits[:, -1, :], sampler, top_p, temperature)

        while generated.shape[1] < max_total:
            next_ids = torch.where(done, torch.full_like(next_ids, EOS_ID), next_ids)
            generated = torch.cat([generated, next_ids.unsqueeze(1)], dim=1)
            done = done | (next_ids == EOS_ID)
            if done.all():
                break
            logits = self.forward(next_ids.unsqueeze(1), caches=caches)  # one new token at a time
            next_ids = self._sample(logits[:, -1, :], sampler, top_p, temperature)

        self.train()
        return generated

    @staticmethod
    def _sample(logits, sampler, top_p, temperature):
        if sampler == "greedy":
            return logits.argmax(dim=-1)
        probs = F.softmax(logits / temperature, dim=-1)
        sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
        cumulative = sorted_probs.cumsum(dim=-1)
        cutoff = (cumulative <= top_p).sum(dim=-1, keepdim=True).clamp(min=1)
        mask = torch.arange(sorted_probs.shape[-1], device=logits.device)[None, :] < cutoff
        sorted_probs = sorted_probs * mask
        sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
        picked = torch.multinomial(sorted_probs, num_samples=1).squeeze(-1)
        return sorted_idx.gather(-1, picked.unsqueeze(-1)).squeeze(-1)


def build_model(config):
    """Contract function: the one place that turns a config dict into a
    model instance, so every later phase constructs Model A the same way.
    """
    return MiniGPT(config)


USE_GRAD_CHECKPOINTING = True
EOS_ID = None

In [ ]:
%%writefile /kaggle/working/b_model.py
"""b_model.py -- Model B (local/global decoder) classes, copied from p1_04_model_B.ipynb."""
import math
Z_LOSS_COEF = 1e-4
EOS_ID = None
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        norm = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (norm * self.weight.float()).to(dtype)


def build_rope_cache(seq_len, d_head, theta=10000.0, device="cpu"):
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_head, 2, device=device).float() / d_head))
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.outer(t, inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    x1, x2 = x[..., 0::2], x[..., 1::2]
    cos = cos[None, None, :, :].to(x.dtype)
    sin = sin[None, None, :, :].to(x.dtype)
    rotated = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return rotated.flatten(-2)


_TRAIN_MASK_CACHE = {}


def sliding_window_mask(q_start, n_q, n_k, window, device, cacheable=False):
    """Boolean [n_q, n_k] mask, True = may attend. Query i sits at absolute
    position q_start + i; key j at position j. Allowed iff the key is not in
    the future AND is less than `window` positions back.
    Only training masks are cached (training always uses one fixed shape);
    eval/generation shapes vary per prompt, so caching them would leak memory.
    """
    cache_key = (n_q, n_k, window, str(device))
    if cacheable and cache_key in _TRAIN_MASK_CACHE:
        return _TRAIN_MASK_CACHE[cache_key]
    q_pos = torch.arange(q_start, q_start + n_q, device=device)[:, None]
    k_pos = torch.arange(n_k, device=device)[None, :]
    allowed = (k_pos <= q_pos) & (q_pos - k_pos < window)
    if cacheable:
        _TRAIN_MASK_CACHE[cache_key] = allowed
    return allowed


class LocalGlobalAttention(nn.Module):
    """MQA attention. Local layers: RoPE + sliding-window mask. Global
    layers: no RoPE (NoPE), plain causal attention over everything.
    """
    def __init__(self, d_model, n_head, n_kv_head, d_head, use_rope, qk_norm):
        super().__init__()
        self.n_head, self.n_kv_head, self.d_head = n_head, n_kv_head, d_head
        self.use_rope = use_rope
        self.q_proj = nn.Linear(d_model, n_head * d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_head * d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_head * d_head, bias=False)
        self.o_proj = nn.Linear(n_head * d_head, d_model, bias=False)
        self.q_norm = RMSNorm(d_head) if qk_norm else None
        self.k_norm = RMSNorm(d_head) if qk_norm else None

    def forward(self, x, cos, sin, attn_mask, cache=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        if self.q_norm is not None:
            q, k = self.q_norm(q), self.k_norm(k)
        if self.use_rope:
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if cache is not None:
            k, v = cache.update(k, v)
        # The cache keeps the single KV head (that's MQA's memory saving);
        # expand it to all query heads only for the attention call itself.
        # An explicit copy keeps SDPA on its fast memory-efficient kernel.
        n_rep = self.n_head // self.n_kv_head
        if n_rep > 1:
            k = k.repeat_interleave(n_rep, dim=1)
            v = v.repeat_interleave(n_rep, dim=1)
        if attn_mask is None:
            # Global layer. Training / prefill: square causal. Decode step
            # (one query): may see every cached key -> no mask needed.
            out = F.scaled_dot_product_attention(q, k, v, is_causal=T > 1)
        else:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)
        out = out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head)
        return self.o_proj(out)


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class Block(nn.Module):
    def __init__(self, cfg, is_global):
        super().__init__()
        self.is_global = is_global
        self.attn_norm = RMSNorm(cfg["d_model"])
        self.attn = LocalGlobalAttention(cfg["d_model"], cfg["n_head"], cfg["n_kv_head"], cfg["d_head"],
                                         use_rope=not is_global, qk_norm=cfg["qk_norm"])
        self.mlp_norm = RMSNorm(cfg["d_model"])
        self.mlp = SwiGLU(cfg["d_model"], cfg["d_ff"])

    def _forward_inner(self, x, cos, sin, attn_mask, cache):
        x = x + self.attn(self.attn_norm(x), cos, sin, attn_mask, cache)
        x = x + self.mlp(self.mlp_norm(x))
        return x

    def forward(self, x, cos, sin, attn_mask, cache=None, use_checkpoint=False):
        if use_checkpoint and cache is None and self.training:
            return grad_checkpoint(self._forward_inner, x, cos, sin, attn_mask, None, use_reentrant=False)
        return self._forward_inner(x, cos, sin, attn_mask, cache)


class KVCache:
    """Pre-allocated per-layer cache. Keeps the full history (with 1 KV
    head it's only ~0.5 MB per layer at 2048 tokens); the sliding window is
    applied by the attention mask, not by evicting entries.
    """
    def __init__(self, batch_size, n_kv_head, max_seq_len, d_head, device, dtype):
        shape = (batch_size, n_kv_head, max_seq_len, d_head)
        self.k = torch.zeros(shape, device=device, dtype=dtype)
        self.v = torch.zeros(shape, device=device, dtype=dtype)
        self.pos = 0

    def update(self, k, v):
        T = k.shape[2]
        self.k[:, :, self.pos:self.pos + T] = k
        self.v[:, :, self.pos:self.pos + T] = v
        self.pos += T
        return self.k[:, :, :self.pos], self.v[:, :, :self.pos]


def is_global_layer(layer_idx, global_every):
    return (layer_idx + 1) % global_every == 0


class LocalGlobalGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed = nn.Embedding(config["vocab_size"], config["d_model"])
        self.blocks = nn.ModuleList([
            Block(config, is_global=is_global_layer(i, config["global_every"]))
            for i in range(config["n_layer"])
        ])
        self.norm_f = RMSNorm(config["d_model"])
        self.head = nn.Linear(config["d_model"], config["vocab_size"], bias=False)
        self.head.weight = self.embed.weight  # tied embeddings

        cos, sin = build_rope_cache(config["max_seq_len"], config["d_head"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config["n_layer"]))

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def layer_pattern(self):
        return "".join("G" if b.is_global else "L" for b in self.blocks)

    def new_kv_caches(self, batch_size, device, dtype):
        return [
            KVCache(batch_size, self.config["n_kv_head"], self.config["max_seq_len"],
                    self.config["d_head"], device, dtype)
            for _ in range(self.config["n_layer"])
        ]

    def _run_blocks(self, x, pos_offset, caches, use_ckpt):
        T = x.shape[1]
        assert pos_offset + T <= self.config["max_seq_len"], "Sequence longer than max_seq_len."
        cos = self.rope_cos[pos_offset:pos_offset + T]
        sin = self.rope_sin[pos_offset:pos_offset + T]
        local_mask = sliding_window_mask(pos_offset, T, pos_offset + T, self.config["window"], x.device,
                                         cacheable=self.training and caches is None)
        for i, blk in enumerate(self.blocks):
            mask = None if blk.is_global else local_mask
            cache = caches[i] if caches is not None else None
            x = blk(x, cos, sin, mask, cache=cache, use_checkpoint=use_ckpt)
        return x

    def encode(self, input_ids, attention_mask=None, mode="causal"):
        """Contract method: hidden states, no LM head. attention_mask is
        accepted for interface compatibility (Phase 1 packs full windows)."""
        assert mode == "causal", "Model B is decoder-only; only causal encoding is supported."
        x = self.embed(input_ids)
        x = self._run_blocks(x, 0, None, use_ckpt=self.training and USE_GRAD_CHECKPOINTING)
        return self.norm_f(x)

    def forward(self, ids, targets=None, caches=None):
        """No targets -> logits. With targets -> tensor [1, 2] = [cross-entropy,
        cross-entropy + z-loss]. The [1, 2] shape is so nn.DataParallel can
        stack one row per GPU; callers take .mean(0).
        In training mode, fp16 autocast is switched on here (inside the
        module) so it also applies inside DataParallel's worker threads.
        """
        use_amp = self.training and ids.is_cuda
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            if caches is None:
                h = self.encode(ids)
            else:
                pos_offset = caches[0].pos
                assert ids.shape[1] == 1 or pos_offset == 0, "Prefill must start from an empty cache."
                h = self.norm_f(self._run_blocks(self.embed(ids), pos_offset, caches, use_ckpt=False))
            logits = self.head(h)
        if targets is None:
            return logits
        logits32 = logits.float()
        lse = torch.logsumexp(logits32, dim=-1)
        target_logit = logits32.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
        ce = (lse - target_logit).mean()           # identical to F.cross_entropy
        total = ce + Z_LOSS_COEF * lse.pow(2).mean()
        return torch.stack([ce, total]).unsqueeze(0)

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, kv_cache=True, sampler="greedy",
                 top_p=0.9, temperature=1.0, repetition_penalty=1.0):
        """Contract method. Stops per sequence at <eos>. Prompts too long to
        leave room for max_new_tokens are cut to their most recent tokens."""
        assert kv_cache, "Only the KV-cache path is implemented."
        was_training = self.training
        self.eval()
        device, dtype = input_ids.device, next(self.parameters()).dtype

        max_prompt_len = self.config["max_seq_len"] - max_new_tokens
        if input_ids.shape[1] > max_prompt_len:
            input_ids = input_ids[:, -max_prompt_len:]

        B, T0 = input_ids.shape
        max_total = min(T0 + max_new_tokens, self.config["max_seq_len"])
        caches = self.new_kv_caches(batch_size=B, device=device, dtype=dtype)
        generated = input_ids
        done = torch.zeros(B, dtype=torch.bool, device=device)

        logits = self.forward(input_ids, caches=caches)[:, -1, :]
        next_ids = self._sample(self._penalize(logits, generated, repetition_penalty), sampler, top_p, temperature)
        while generated.shape[1] < max_total:
            next_ids = torch.where(done, torch.full_like(next_ids, EOS_ID), next_ids)
            generated = torch.cat([generated, next_ids.unsqueeze(1)], dim=1)
            done = done | (next_ids == EOS_ID)
            if done.all() or generated.shape[1] >= max_total:
                break
            logits = self.forward(next_ids.unsqueeze(1), caches=caches)[:, -1, :]
            next_ids = self._sample(self._penalize(logits, generated, repetition_penalty), sampler, top_p, temperature)

        self.train(was_training)
        return generated

    @staticmethod
    def _penalize(logits, previous_ids, penalty):
        """CTRL-style repetition penalty on every token already present."""
        if penalty == 1.0:
            return logits
        scores = logits.gather(1, previous_ids)
        scores = torch.where(scores > 0, scores / penalty, scores * penalty)
        return logits.scatter(1, previous_ids, scores)

    @staticmethod
    def _sample(logits, sampler, top_p, temperature):
        if sampler == "greedy":
            return logits.argmax(dim=-1)
        probs = F.softmax(logits / temperature, dim=-1)
        sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
        cumulative = sorted_probs.cumsum(dim=-1)
        cutoff = (cumulative <= top_p).sum(dim=-1, keepdim=True).clamp(min=1)
        keep = torch.arange(sorted_probs.shape[-1], device=logits.device)[None, :] < cutoff
        sorted_probs = sorted_probs * keep
        sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
        picked = torch.multinomial(sorted_probs, num_samples=1).squeeze(-1)
        return sorted_idx.gather(-1, picked.unsqueeze(-1)).squeeze(-1)


def build_model(config):
    """Contract function."""
    return LocalGlobalGPT(config)


USE_GRAD_CHECKPOINTING = True

In [ ]:
%%writefile /kaggle/working/lsi_lib.py
"""lsi_lib.py -- Phase 5 (Legal Statute Identification, LSI) library.

Task: a legal document -> the SET of statute ids that apply (multi-label). Metric: macro-F1 over
statutes, so rare statutes count as much as frequent ones.

Pipeline (every experiment):
  document -> Phase-1 tokens -> 512-token chunks with 64-token overlap. Documents with more than
  MAX_CHUNKS chunks keep MAX_CHUNKS of them spread evenly (first and last always kept).
  Stage 1  ADAPT   Phase-1 backbone + adaptation method + a label-wise attention head (every statute
                   has its own attention over the chunk's tokens) -> one logit per statute for each
                   chunk. A document is a bag of up to K_TRAIN random chunks;
                   its logits are the log-sum-exp pool of its chunk logits (multiple-instance
                   learning: a statute applies if SOME part of the document supports it).
                   Loss: BCE with a per-statute positive weight. Trained on 80% of train.
  Stage 2  ENCODE  the adapted backbone turns every chunk of the held-out 20% of train, dev and
                   test into a vector (+ its stage-1 statute logits).
  Stage 3  READ    document readers over the chunk vectors, trained on the held-out 20% only (the
                   encoder never saw those documents): mean, max, attention, label-wise attention
                   (one attention query per statute), label-wise attention + statute co-occurrence
                   refinement, transformer + label-wise attention. Plus a loss study (BCE, weighted
                   BCE, focal, asymmetric) and a section study (first chunk, first / middle / last
                   third). The reader is chosen by validation macro-average-precision.
  CALIBRATE        decision thresholds fitted on dev. The strategy (0.5, one global threshold,
                   per-statute thresholds, per-statute shrunk toward the global one) is chosen by an
                   honest 2-fold check on dev; every document gets at least its top statute.

Methods (own code): LoRMA (Bihany et al. 2025), QLoRA (Dettmers et al. 2023: NF4 4-bit frozen
base + double-quantised scales + LoRA), Houlsby adapters (Houlsby et al. 2019), prefix tuning
(Li & Liang 2021, deep form: learned states at every layer), partial / layer-wise fine-tuning,
full fine-tuning. Distillation (Hinton et al. 2015): a half-depth D student (every other encoder
layer, DistilBERT-style init) reads 4 chunks and learns gold statutes + the teachers' statute
probabilities.
"""
import csv
import json
import math
import random
import re
import time
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

CHUNK = 512
STRIDE = 448                     # 64-token overlap
MAX_CHUNKS = 12                  # chunks kept per document (evenly spread)
K_TRAIN = 3                      # chunks per document per stage-1 step
KD_CHUNKS = 4                    # chunks the distillation student reads (evenly spread)
LSE_R = 4.0                      # sharpness of log-sum-exp pooling (0 = mean, inf = max)
SHRINK_K = 8.0                   # per-statute threshold shrinkage toward the global threshold
LOSSES = ("bce", "wbce", "focal", "asl")


# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------
def load_json_any(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except json.JSONDecodeError:
        with open(path, encoding="utf-8") as f:
            return [json.loads(line) for line in f if line.strip()]


def _first(d, keys, default=None):
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    return default


def parse_labels(v):
    """Statute ids -> sorted unique ints (accepts ints, '12', 12.0, a single id or a list)."""
    if v is None:
        return None
    if not isinstance(v, (list, tuple)):
        v = [v]
    out = []
    for x in v:
        if isinstance(x, str):
            x = x.strip()
            assert re.fullmatch(r"-?\d+(\.0+)?", x), f"statute id {x!r} is not an integer"
        out.append(int(float(x)))
    return sorted(set(out))


def parse_docs(obj):
    """A list of documents, a {id: document} dict or {"data": [...]} ->
    list of {"id": str, "raw_id": as given, "text": str, "labels": sorted ints or None}.
    'text' may be a string or a list of sentences."""
    if isinstance(obj, dict):
        for k in ("data", "documents", "docs", "cases", "examples", "items"):
            if k in obj and isinstance(obj[k], list):
                return parse_docs(obj[k])
        items = [dict(v, id=v.get("id", k)) if isinstance(v, dict) else {"id": k, "text": v} for k, v in obj.items()]
        return parse_docs(items)
    out = []
    for i, v in enumerate(obj):
        raw = _first(v, ("id", "doc_id", "case_id", "name"), i)
        text = _first(v, ("text", "document", "facts", "case_text", "content"), "")
        if isinstance(text, list):
            text = " ".join(str(t) for t in text)
        lab = _first(v, ("labels", "label", "statutes", "statute_ids", "targets"))
        out.append({"id": str(raw), "raw_id": raw, "text": " ".join(str(text or "").split()), "labels": parse_labels(lab)})
    return out


def _h(t):
    import hashlib
    return hashlib.md5(t.encode("utf-8")).hexdigest()


def clean_splits(docs, log=print):
    """docs = {"train", "dev", "test"} parsed lists. Returns cleaned copies + a report.
    test: ids must be unique (one prediction per id); exact repeats are dropped.
    Internal ids become "<split>:<id>" (raw_id keeps the original for the submission).
    dev:  exact repeats (id, text, labels) dropped; same id with different content -> id#2, id#3.
    train: documents whose text is in dev/test dropped (no leakage); repeated texts merged into one
           document (union of their statutes)."""
    rep = {}
    test, seen = [], {}
    rep["test_rows"], rep["test_repeats_dropped"], rep["test_id_clashes"] = len(docs["test"]), 0, 0
    for d in docs["test"]:
        if d["id"] in seen:
            rep["test_repeats_dropped"] += 1
            if seen[d["id"]] != _h(d["text"]):
                rep["test_id_clashes"] += 1
            continue
        seen[d["id"]] = _h(d["text"])
        test.append(d)
    if rep["test_id_clashes"]:
        log(f"WARNING: {rep['test_id_clashes']} test ids repeat with DIFFERENT text; the first copy is predicted")
    dev, seen_dev, n_used = [], set(), {}
    rep["dev_rows"], rep["dev_repeats_dropped"] = len(docs["dev"]), 0
    for d in docs["dev"]:
        key = (d["id"], _h(d["text"]), tuple(d["labels"] or ()))
        if key in seen_dev:
            rep["dev_repeats_dropped"] += 1
            continue
        seen_dev.add(key)
        n = n_used.get(d["id"], 0)
        n_used[d["id"]] = n + 1
        dev.append(dict(d, id=f"{d['id']}#{n + 1}") if n else d)
    rep["dev_renamed"] = sum(1 for d in dev if "#" in d["id"])
    held = {_h(d["text"]) for d in dev + test}
    first, rep["train_rows"], rep["train_leak_dropped"], rep["train_dup_merged"], rep["train_dup_label_differs"] = {}, len(docs["train"]), 0, 0, 0
    for d in docs["train"]:
        k = _h(d["text"])
        if k in held:
            rep["train_leak_dropped"] += 1
            continue
        if k in first:
            rep["train_dup_merged"] += 1
            old = first[k]
            if (old["labels"] or []) != (d["labels"] or []):
                rep["train_dup_label_differs"] += 1
                first[k] = dict(old, labels=sorted(set(old["labels"] or []) | set(d["labels"] or [])))
            continue
        first[k] = d
    train, ids = [], {}
    for d in first.values():
        n = ids.get(d["id"], 0)
        ids[d["id"]] = n + 1
        train.append(dict(d, id=f"{d['id']}#{n + 1}") if n else d)
    rep["train_docs"], rep["dev_docs"], rep["test_docs"] = len(train), len(dev), len(test)
    # internal ids carry their split ("dev:case_001"), so ids that repeat ACROSS splits can never mix
    # up cached vectors or labels; the submission uses raw_id.
    out = {s: [dict(d, id=f"{s}:{d['id']}") for d in part] for s, part in (("train", train), ("dev", dev), ("test", test))}
    return out, rep


def label_space(train, dev):
    """Sorted statute ids seen in train or dev (the only ids a model can output)."""
    return sorted({l for d in train + dev for l in (d["labels"] or [])})


def windows(n, size, stride):
    if n <= size:
        return [(0, n)]
    starts = list(range(0, n - size + 1, stride))
    if starts[-1] + size < n:
        starts.append(n - size)
    return [(s, s + size) for s in starts]


def select_spans(n):
    spans = windows(n, CHUNK, STRIDE)
    if len(spans) > MAX_CHUNKS:
        idx = np.unique(np.round(np.linspace(0, len(spans) - 1, MAX_CHUNKS)).astype(int))
        spans = [spans[i] for i in idx]
    return spans


def tokenize_docs(docs, encode_batch, fallback_id, label_index, batch=256, log=print):
    """-> list of {"id", "raw_id", "y" (label indices or None), "toks" (uint16, the kept chunks
    one after another), "chunks" [(start, end, start_frac, end_frac)], "n_tok", "n_win"}."""
    out, t0 = [], time.time()
    for i in range(0, len(docs), batch):
        part = docs[i:i + batch]
        toks = encode_batch([d["text"] for d in part])
        for d, t in zip(part, toks):
            t = list(t) or [fallback_id]
            n = len(t)
            spans = select_spans(n)
            kept, chunks, off = [], [], 0
            for s, e in spans:
                kept += t[s:e]
                chunks.append((off, off + e - s, s / n, e / n))
                off += e - s
            arr = np.asarray(kept, dtype=np.int64)
            assert arr.max() < 65536 and arr.min() >= 0, "token id does not fit uint16"
            y = None if d["labels"] is None else sorted({label_index[l] for l in d["labels"] if l in label_index})
            out.append({"id": d["id"], "raw_id": d["raw_id"], "y": y, "toks": arr.astype(np.uint16), "chunks": chunks,
                        "n_tok": n, "n_win": len(windows(n, CHUNK, STRIDE))})
        if (i // batch) % 40 == 0:
            log(f"  tokenized {min(i + batch, len(docs))}/{len(docs)} ({time.time() - t0:.0f}s)")
    return out


def split_train(docs, frac_agg=0.2, frac_val=0.15, seed=7):
    """Train -> adapt (stage 1) / agg_train / agg_val (stage 3). Stratified by each document's
    RAREST statute, so rare statutes reach every part when they can. docs: dicts with "id", "y"."""
    freq = Counter(l for d in docs for l in (d["y"] or []))
    strata = defaultdict(list)
    for d in docs:
        key = min(d["y"], key=lambda l: (freq[l], l)) if d["y"] else -1
        strata[key].append(d["id"])
    rng = random.Random(seed)
    adapt, agg_tr, agg_val = [], [], []
    for key in sorted(strata):
        ids = sorted(strata[key])
        rng.shuffle(ids)
        n_agg = int(frac_agg * len(ids) + rng.random())
        n_val = int(frac_val * n_agg + rng.random())
        agg_val += ids[:n_val]
        agg_tr += ids[n_val:n_agg]
        adapt += ids[n_agg:]
    return {"adapt": sorted(adapt), "agg_train": sorted(agg_tr), "agg_val": sorted(agg_val)}


def kd_view(doc):
    n = len(doc["chunks"])
    if n <= KD_CHUNKS:
        return list(range(n))
    return [int(i) for i in np.unique(np.round(np.linspace(0, n - 1, KD_CHUNKS)).astype(int))]


def region_select(doc, region):
    """Chunk indices whose centre lies in the region; 'first' = the first chunk only."""
    if region == "first":
        return [0]
    lo, hi = {"begin": (0, 1 / 3), "middle": (1 / 3, 2 / 3), "end": (2 / 3, 1.0001)}[region]
    cent = [(fs + fe) / 2 for _, _, fs, fe in doc["chunks"]]
    sel = [j for j, c in enumerate(cent) if lo <= c < hi]
    return sel or [int(np.argmin([min(abs(c - lo), abs(c - hi)) for c in cent]))]


def label_matrix(docs, L):
    Y = np.zeros((len(docs), L), dtype=bool)
    for i, d in enumerate(docs):
        if d["y"]:
            Y[i, d["y"]] = True
    return Y


# ---------------------------------------------------------------------------
# Metrics and thresholds
# ---------------------------------------------------------------------------
def multilabel_report(Y, P):
    """Y, P bool [n, L]. Macro scores average over the statutes that occur in Y (the evaluator
    derives the label set from the reference); micro-F1 and subset accuracy on the same columns."""
    Y, P = np.asarray(Y, bool), np.asarray(P, bool)
    cols = Y.any(0)
    Y, P = Y[:, cols], P[:, cols]
    tp, fp, fn = (Y & P).sum(0), (~Y & P).sum(0), (Y & ~P).sum(0)
    f1 = 2 * tp / np.maximum(1, 2 * tp + fp + fn)
    prec = tp / np.maximum(1, tp + fp)
    rec = tp / np.maximum(1, tp + fn)
    TP, FP, FN = tp.sum(), fp.sum(), fn.sum()
    if not cols.any():
        return {"macro_f1": 0.0, "macro_precision": 0.0, "macro_recall": 0.0, "micro_f1": 0.0, "subset_accuracy": 0.0, "labels_scored": 0}
    return {"macro_f1": round(float(f1.mean()), 4), "macro_precision": round(float(prec.mean()), 4),
            "macro_recall": round(float(rec.mean()), 4), "micro_f1": round(float(2 * TP / max(1, 2 * TP + FP + FN)), 4),
            "subset_accuracy": round(float((Y == P).all(1).mean()), 4), "labels_scored": int(cols.sum()),
            "avg_predicted_per_doc": round(float(P.sum(1).mean()), 3)}


def per_label_f1(Y, P):
    tp, fp, fn = (Y & P).sum(0), (~Y & P).sum(0), (Y & ~P).sum(0)
    return 2 * tp / np.maximum(1, 2 * tp + fp + fn), tp, fp, fn


def average_precision(y, s):
    y = np.asarray(y, bool)
    if not y.any():
        return float("nan")
    order = np.argsort(-np.asarray(s, float), kind="mergesort")
    yy = y[order]
    tp = np.cumsum(yy)
    prec = tp / np.arange(1, len(yy) + 1)
    return float((prec * yy).sum() / yy.sum())


def macro_ap(Y, S):
    aps = [average_precision(Y[:, l], S[:, l]) for l in range(Y.shape[1]) if Y[:, l].any()]
    return round(float(np.mean(aps)), 4) if aps else 0.0


def apply_thresholds(S, thr):
    """S logits [n, L], thr [L] -> bool predictions; a document with nothing above its
    thresholds gets its single best statute (every document has at least one statute)."""
    P = S > thr[None, :]
    empty = ~P.any(1)
    if empty.any():
        rows = np.where(empty)[0]
        P[rows, np.argmax(S[rows] - thr[None, :], 1)] = True
    return P


def best_label_thresholds(S, Y):
    """Per statute: the threshold with the best F1 on (S, Y); nan where Y has no positive."""
    n, L = S.shape
    thr = np.full(L, np.nan)
    for l in range(L):
        y = Y[:, l]
        npos = int(y.sum())
        if npos == 0:
            continue
        order = np.argsort(-S[:, l], kind="mergesort")
        s, yy = S[order, l], y[order]
        tp = np.cumsum(yy)
        k = np.arange(1, n + 1)
        f1 = 2 * tp / (k + npos)                   # 2tp / (2tp + fp + fn), fp = k - tp, fn = npos - tp
        valid = np.r_[s[:-1] > s[1:], True]       # never cut between tied scores
        j = int(np.argmax(np.where(valid, f1, -1)))
        thr[l] = (s[j] + s[j + 1]) / 2 if j + 1 < n else s[j] - 1e-3
    return thr


def best_global_threshold(S, Y):
    cols = Y.any(0)
    cands = np.unique(np.r_[0.0, np.quantile(S, np.linspace(0.5, 0.9995, 240))])
    best, best_t = -1.0, 0.0
    Ys = Y[:, cols]
    for t in cands:
        P = apply_thresholds(S, np.full(S.shape[1], t))[:, cols]
        tp, fp, fn = (Ys & P).sum(0), (~Ys & P).sum(0), (Ys & ~P).sum(0)
        f = float((2 * tp / np.maximum(1, 2 * tp + fp + fn)).mean())
        if f > best:
            best, best_t = f, float(t)
    return best_t


THRESHOLD_STRATEGIES = ("prob_0.5", "global", "per_label", "per_label_shrunk")


def fit_thresholds(strategy, S, Y, never=None):
    """never: statutes with no training document -- never predicted (a dev-fitted threshold for
    them would only chase dev noise)."""
    L = S.shape[1]
    if strategy == "prob_0.5":
        thr = np.zeros(L)
    else:
        tg = best_global_threshold(S, Y)
        tl = best_label_thresholds(S, Y)
        npos = Y.sum(0)
        w = npos / (npos + SHRINK_K)
        thr = {"global": np.full(L, tg), "per_label": np.where(np.isnan(tl), tg, tl),
               "per_label_shrunk": np.where(np.isnan(tl), tg, w * np.nan_to_num(tl) + (1 - w) * tg)}[strategy]
    if never is not None:
        thr = np.where(never, np.inf, thr)
    return thr


def unseen(data):
    """Statutes with no training document."""
    return label_matrix(data["train"], len(data["labels"])).sum(0) == 0


def calibrate(S, Y, seed=0, strategies=THRESHOLD_STRATEGIES, never=None):
    """Choose the threshold strategy by an honest 2-fold check on dev (fit on one half, score the
    other, both ways), then fit the chosen strategy on all of dev."""
    S, Y = np.asarray(S, float), np.asarray(Y, bool)
    rng = np.random.RandomState(seed)
    a = np.zeros(len(S), bool)
    a[rng.permutation(len(S))[: len(S) // 2]] = True
    res = {}
    for name in strategies:
        P = np.zeros_like(Y)
        if a.any() and (~a).any():
            P[~a] = apply_thresholds(S[~a], fit_thresholds(name, S[a], Y[a], never))
            P[a] = apply_thresholds(S[a], fit_thresholds(name, S[~a], Y[~a], never))
        else:
            P = apply_thresholds(S, fit_thresholds(name, S, Y, never))
        res[name] = multilabel_report(Y, P)
    chosen = max(strategies, key=lambda k: res[k]["macro_f1"])
    thr = fit_thresholds(chosen, S, Y, never)
    return thr, {"strategy_2fold": res, "chosen_strategy": chosen}


def example_f1(Y, P):
    inter = (Y & P).sum(1)
    return 2 * inter / np.maximum(1, Y.sum(1) + P.sum(1))


# ---------------------------------------------------------------------------
# Adaptation methods
# ---------------------------------------------------------------------------
LINEAR_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")
ADAPTER_TARGETS = ("o_proj", "down_proj")      # the output of each sublayer (Houlsby positions)


class LoRMALinear(nn.Module):
    """LoRMA, additive-identity form: y = (I + s B A) W0 x = h + s B A h with h = W0 x.
    B = 0 at the start, so the layer starts exactly as the Phase-1 layer."""

    def __init__(self, base, r, alpha):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scale = alpha / r

    def forward(self, x):
        h = self.base(x)
        return h + F.linear(F.linear(h, self.A), self.B) * self.scale


NF4_VALUES = (-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635,
              -0.18477343022823334, -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725,
              0.24611230194568634, 0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
              0.7229568362236023, 1.0)


def quantize_nf4(w, block=64, dq_block=256):
    """4-bit NormalFloat (QLoRA): blocks of `block` weights scaled by their absmax, each weight
    -> nearest of 16 normal quantiles (2 codes per byte). Double quantisation: the block absmax
    values are themselves stored as int8 (minus their mean, scaled per `dq_block` blocks)."""
    flat = w.detach().float().flatten()
    n = flat.numel()
    flat = F.pad(flat, (0, (-n) % (2 * block)))
    blocks = flat.view(-1, block)
    absmax = blocks.abs().amax(1).clamp(min=1e-12)
    table = torch.tensor(NF4_VALUES)
    idx = torch.bucketize((blocks / absmax[:, None]).flatten(), (table[1:] + table[:-1]) / 2)
    codes = ((idx[0::2] << 4) | idx[1::2]).to(torch.uint8)
    c = absmax.mean()
    a = absmax - c
    nb = a.numel()
    a = F.pad(a, (0, (-nb) % dq_block)).view(-1, dq_block)
    s2 = a.abs().amax(1).clamp(min=1e-12)
    qabs = torch.round(a / s2[:, None] * 127).clamp(-127, 127).to(torch.int8).flatten()
    return {"codes": codes, "qabs": qabs, "s2": s2.float(), "c": c.float().reshape(1), "n": n, "nb": nb,
            "shape": tuple(w.shape), "block": block, "dq_block": dq_block}


def dequantize_nf4(codes, qabs, s2, c, table, n, nb, shape, block, dq_block, dtype=torch.float32):
    idx = torch.stack([(codes >> 4).long(), (codes & 15).long()], 1).flatten()
    absmax = ((qabs.view(-1, dq_block).float() / 127) * s2[:, None]).flatten()[:nb] + c
    w = (table[idx].view(-1, block).float() * absmax[:, None]).flatten()[:n]
    return w.view(shape).to(dtype)


class QLoRALinear(nn.Module):
    """QLoRA: frozen base weight stored in NF4 (4-bit) + double-quantised scales, de-quantised on
    the fly; trainable LoRA y = W_nf4 x + s B A x (B = 0, so it starts as the 4-bit model)."""

    def __init__(self, base, r, alpha, block=64, dq_block=256):
        super().__init__()
        q = quantize_nf4(base.weight, block, dq_block)
        self.out_features, self.in_features = base.out_features, base.in_features
        for k in ("codes", "qabs", "s2", "c"):
            self.register_buffer(k, q[k])
        self.register_buffer("table", torch.tensor(NF4_VALUES))
        self.meta = (q["n"], q["nb"], q["shape"], block, dq_block)
        with torch.no_grad():                       # how far the 4-bit weight is from the Phase-1 weight
            w = base.weight.detach().float()
            self._err = float((self.weight() - w).norm() / w.norm().clamp(min=1e-12))
        self.bias = None if base.bias is None else nn.Parameter(base.bias.detach().clone(), requires_grad=False)
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scale = alpha / r

    def weight(self, dtype=torch.float32):
        return dequantize_nf4(self.codes, self.qabs, self.s2, self.c, self.table, *self.meta, dtype=dtype)

    def forward(self, x):
        W = self.weight(torch.float16 if x.is_cuda else torch.float32)
        y = F.linear(x, W, self.bias)
        return y + F.linear(F.linear(x, self.A), self.B) * self.scale


class AdapterLinear(nn.Module):
    """Houlsby adapter on a sublayer output (before the residual add): y = h + U gelu(D h), h = W0 x.
    U = 0 at the start, so the layer starts exactly as the Phase-1 layer."""

    def __init__(self, base, m):
        super().__init__()
        self.base = base
        self.down = nn.Linear(base.out_features, m)
        self.up = nn.Linear(m, base.out_features)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        h = self.base(x)
        return h + self.up(F.gelu(self.down(h)))


class PrefixTuner(nn.Module):
    """Prefix tuning, deep form (Li & Liang 2021; P-Tuning v2): P extra positions at the front of
    every chunk whose hidden state at the INPUT OF EVERY BLOCK is a learned vector. Every real token
    can attend to them at every layer (their keys/values come from the learned states through the
    frozen projections). Implemented with forward pre-hooks, so no model code changes."""

    def __init__(self, blocks, n_prefix, d):
        super().__init__()
        self.P = n_prefix
        self.prefix = nn.Parameter(torch.zeros(len(blocks), n_prefix, d))
        self.capture = None
        self.handles = [b.register_forward_pre_hook(self._hook(i)) for i, b in enumerate(blocks)]

    def _hook(self, i):
        def hook(module, args):
            x = args[0]
            if self.capture is not None:            # initialisation pass: record real activations
                v = x[:, self.P:2 * self.P].detach().float().mean(0)
                if v.shape[0] < self.P:
                    v = torch.cat([v, v.mean(0, keepdim=True).expand(self.P - v.shape[0], -1)])
                self.capture[i] = v
                return None
            p = self.prefix[i].to(x.dtype).unsqueeze(0).expand(x.shape[0], -1, -1)
            return (torch.cat([p, x[:, self.P:]], 1),) + tuple(args[1:])
        return hook

    def remove(self):
        for h in self.handles:
            h.remove()


def _replace(module, name, new):
    parent = module
    parts = name.split(".")
    for p in parts[:-1]:
        parent = getattr(parent, p)
    setattr(parent, parts[-1], new)


def blocks_of(kind, model):
    return model.enc_blocks if kind in ("D", "Ds") else model.blocks


def final_norm(kind, model):
    return model.enc_norm if kind in ("D", "Ds") else model.norm_f


def max_len_of(kind, model):
    cfg = getattr(model, "config", {}) or {}
    return cfg.get("max_enc_len") if kind in ("D", "Ds") else cfg.get("max_seq_len")


def norm_params(model):
    return [p for m in model.modules() if type(m).__name__.endswith("Norm") for p in m.parameters(recurse=False)]


def apply_method(model, kind, method, exp):
    """Freeze the Phase-1 backbone, then insert / open what the method trains.
    Returns (backbone-side trainable parameters, PrefixTuner or None)."""
    for p in model.parameters():
        p.requires_grad_(False)
    tuner = None
    if method == "full":
        for p in model.parameters():
            p.requires_grad_(True)
    elif method == "partial":
        blocks = list(blocks_of(kind, model))
        for b in blocks[-exp["top_k"]:]:
            for p in b.parameters():
                p.requires_grad_(True)
        for p in final_norm(kind, model).parameters():
            p.requires_grad_(True)
    elif method in ("lorma", "qlora", "adapter"):
        targets = ADAPTER_TARGETS if method == "adapter" else LINEAR_TARGETS
        names = [n for n, m in model.named_modules() if isinstance(m, nn.Linear) and n.split(".")[-1] in targets]
        assert names, "no target linear layers found"
        for n in names:
            lin = model.get_submodule(n)
            if method == "lorma":
                new = LoRMALinear(lin, exp.get("r", 16), exp.get("alpha", 32))
            elif method == "qlora":
                new = QLoRALinear(lin, exp.get("r", 16), exp.get("alpha", 32))
            else:
                new = AdapterLinear(lin, exp.get("bottleneck", 64))
            _replace(model, n, new)
        if method == "adapter":                    # Houlsby et al. also train the layer norms
            for p in norm_params(model):
                p.requires_grad_(True)
    elif method == "prefix":
        d = exp["d_model"]
        tuner = PrefixTuner(list(blocks_of(kind, model)), exp.get("n_prefix", 16), d)
    else:
        raise ValueError(method)
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return n + (tuner.prefix.numel() if tuner is not None else 0), tuner


def storage_mb(model):
    """Deployable backbone size: fp16 parameters + packed 4-bit buffers (QLoRA)."""
    b = sum(p.numel() * 2 for p in model.parameters())
    b += sum(buf.numel() * buf.element_size() for n, buf in model.named_buffers() if n.split(".")[-1] in ("codes", "qabs", "s2", "c"))
    return round(b / 2 ** 20, 1)


# ---------------------------------------------------------------------------
# Encoders, stage-1 model, readers
# ---------------------------------------------------------------------------
class AttnPool(nn.Module):
    """Attention pooling over tokens; the query starts at 0, i.e. as mean pooling."""

    def __init__(self, d):
        super().__init__()
        self.q = nn.Parameter(torch.zeros(d))

    def forward(self, h, mask):
        s = (h @ self.q) / math.sqrt(h.shape[-1])
        w = torch.softmax(s.masked_fill(~mask, -1e4), -1)
        return (w.unsqueeze(-1) * h).sum(1)


class ChunkEncoder(nn.Module):
    """token chunk [B, T] (right-padded) -> chunk vector [B, d]."""

    def __init__(self, kind, model, d, tuner=None, prefix_id=0):
        super().__init__()
        self.kind, self.m, self.d = ("D" if kind == "Ds" else kind), model, d
        self.tuner, self.prefix_id = tuner, prefix_id
        self.P = tuner.P if tuner is not None else 0
        self.max_len = max_len_of(kind, model)
        self.pool = AttnPool(d)

    def states(self, ids, mask):
        P = self.P
        if P:
            if self.max_len and ids.shape[1] + P > self.max_len:
                ids, mask = ids[:, :self.max_len - P], mask[:, :self.max_len - P]
            B = ids.shape[0]
            ids = torch.cat([ids.new_full((B, P), self.prefix_id), ids], 1)
            full = torch.cat([mask.new_ones((B, P)), mask], 1)
        else:
            full = mask
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=ids.is_cuda):
            if self.kind == "D":
                h = self.m.encode(ids, full)
            elif self.kind == "C":
                h = self.m.encode(ids, mode="prefix", prefix_len=full.sum(1))
            else:                                   # A, B: left to right
                h = self.m.encode(ids)
        h = h.float()
        return (h[:, P:], mask) if P else (h, mask)

    def forward(self, ids, mask):
        h, m = self.states(ids, mask)
        return self.pool(h, m)

    @torch.no_grad()
    def init_prefix(self, ids, mask):
        """Li & Liang: start the prefix from activations of real words (here: the first P tokens
        of real training chunks, at every layer)."""
        if self.tuner is None:
            return
        was = self.training
        self.eval()
        self.tuner.capture = {}
        self.states(ids, mask)
        for i, v in self.tuner.capture.items():
            self.tuner.prefix.data[i] = v.to(self.tuner.prefix.dtype)
        self.tuner.capture = None
        self.train(was)


def prior_logits(Y):
    p = (Y.sum(0) + 1.0) / (len(Y) + 2.0)
    return torch.tensor(np.log(p / (1 - p)), dtype=torch.float32)


class LabelAttnHead(nn.Module):
    """Label-wise attention over TOKENS (CAML / LAAT style): every statute has its own query and
    reads the tokens that matter for it; logit_l = w_l . sum_t a_lt h_t + b_l. Queries start at 0
    (= mean of the tokens), biases at the statute's log prior."""

    def __init__(self, d, n_labels, prior):
        super().__init__()
        self.out_features = n_labels
        self.Q = nn.Parameter(torch.zeros(n_labels, d))
        self.W = nn.Parameter(torch.randn(n_labels, d) * 0.02)
        self.b = nn.Parameter(prior.clone())

    def forward(self, h, mask):
        s = (h @ self.Q.t()) / math.sqrt(h.shape[-1])                  # [B, T, L]
        a = torch.softmax(s.masked_fill(~mask.unsqueeze(-1), -1e4), 1)
        return (torch.einsum("btl,btd->bld", a, h) * self.W).sum(-1) + self.b


class ChunkModel(nn.Module):
    """Stage-1 model: chunk -> (statute logits from the label-wise token attention head,
    chunk vector from attention pooling -- the vector the stage-3 readers use)."""

    def __init__(self, encoder, d, n_labels, prior, drop=0.1):
        super().__init__()
        self.enc, self.drop, self.out = encoder, nn.Dropout(drop), LabelAttnHead(d, n_labels, prior)

    def forward(self, ids, mask):
        h, m = self.enc.states(ids, mask)
        return self.out(self.drop(h), m), self.enc.pool(h, m)


def lse_pool(z, owners, n_docs, r=LSE_R):
    """Chunk logits z [n, L] -> document logits [n_docs, L]: (1/r) log mean_j exp(r z_j)."""
    owners = [int(o) for o in owners]
    seen, slot = [0] * n_docs, []
    for o in owners:
        slot.append(seen[o])
        seen[o] += 1
    Z = torch.full((n_docs, max(1, max(seen)), z.shape[1]), -1e4, dtype=z.dtype, device=z.device)
    Z[torch.tensor(owners, device=z.device), torch.tensor(slot, device=z.device)] = r * z
    counts = torch.tensor(seen, dtype=z.dtype, device=z.device).clamp(min=1)
    return (torch.logsumexp(Z, 1) - torch.log(counts)[:, None]) / r


def ml_loss(z, y, kind="wbce", pw=None, gamma=2.0, gamma_neg=4.0, clip=0.05):
    z = z.float()
    if kind == "bce":
        return F.binary_cross_entropy_with_logits(z, y)
    if kind == "wbce":
        return F.binary_cross_entropy_with_logits(z, y, pos_weight=pw)
    if kind == "focal":                            # Lin et al. 2017
        ce = F.binary_cross_entropy_with_logits(z, y, reduction="none")
        p = torch.sigmoid(z)
        pt = p * y + (1 - p) * (1 - y)
        return ((1 - pt) ** gamma * ce).mean()
    if kind == "asl":                              # Ridnik et al. 2021: gamma+ = 0, gamma- = 4, clip 0.05
        p = torch.sigmoid(z)
        pm = (p - clip).clamp(min=0)
        pos = y * F.logsigmoid(z)
        neg = (1 - y) * pm ** gamma_neg * torch.log((1 - pm).clamp(min=1e-8))
        return -(pos + neg).mean()
    raise ValueError(kind)


class Reader(nn.Module):
    """Document reader over chunk vectors X [B, n, d], positions P [B, n, 4], mask M [B, n] ->
    statute logits [B, L] (+ attention weights). 'lwan' = label-wise attention: every statute has
    its own query over the chunks (LAAT / CAML style); 'lwan_cooc' adds a learned statute-
    co-occurrence correction z + R sigmoid(z) (R starts at 0)."""

    KINDS = ("mean", "max", "attn", "lwan", "lwan_cooc", "tr_lwan")

    def __init__(self, d, n_labels, kind, prior, h=256, drop=0.2):
        super().__init__()
        self.kind = kind
        self.inp = nn.Sequential(nn.LayerNorm(d), nn.Dropout(drop), nn.Linear(d, h), nn.GELU())
        self.pos = nn.Linear(4, h)
        if kind == "tr_lwan":
            layer = nn.TransformerEncoderLayer(h, 4, 2 * h, drop, batch_first=True, norm_first=True)
            self.tr = nn.TransformerEncoder(layer, 2, enable_nested_tensor=False)
        if kind == "attn":
            self.score = nn.Linear(h, 1)
        if kind in ("mean", "max", "attn"):
            self.out = nn.Sequential(nn.Dropout(drop), nn.Linear(h, n_labels))
            with torch.no_grad():
                self.out[1].bias.copy_(prior)
        else:
            self.Q = nn.Parameter(torch.randn(n_labels, h) * 0.02)
            self.W = nn.Parameter(torch.randn(n_labels, h) * 0.02)
            self.b = nn.Parameter(prior.clone())
            self.drop = nn.Dropout(drop)
        if kind == "lwan_cooc":
            self.R = nn.Parameter(torch.zeros(n_labels, n_labels))
            self.register_buffer("off", 1 - torch.eye(n_labels))

    def forward(self, X, P, M):
        x = self.inp(X) + self.pos(P)
        if self.kind == "tr_lwan":
            x = self.tr(x, src_key_padding_mask=~M)
        if self.kind == "mean":
            m = M.unsqueeze(-1).float()
            return self.out((x * m).sum(1) / m.sum(1).clamp(min=1)), None
        if self.kind == "max":
            return self.out(x.masked_fill(~M.unsqueeze(-1), -1e4).max(1).values), None
        if self.kind == "attn":
            w = torch.softmax(self.score(x).squeeze(-1).masked_fill(~M, -1e4), -1)
            return self.out((w.unsqueeze(-1) * x).sum(1)), w
        s = (x @ self.Q.t()).masked_fill(~M.unsqueeze(-1), -1e4)          # [B, n, L]
        a = torch.softmax(s, 1)
        ctx = torch.einsum("bnl,bnh->blh", a, self.drop(x))
        z = (ctx * self.W).sum(-1) + self.b
        if self.kind == "lwan_cooc":
            z = z + torch.sigmoid(z) @ (self.R * self.off).t()
        return z, a


def pad_tokens(seqs, pad_id, device):
    L = max(1, max(len(s) for s in seqs))
    ids = torch.full((len(seqs), L), pad_id, dtype=torch.long)
    mask = torch.zeros(len(seqs), L, dtype=torch.bool)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.from_numpy(np.asarray(s, dtype=np.int64))
        mask[i, :len(s)] = True
    return ids.to(device), mask.to(device)


def pos_feats(doc, j):
    s, e, fs, fe = doc["chunks"][j]
    return [fs, fe, float(j == 0), float(j == len(doc["chunks"]) - 1)]


def lr_mult(p, warm=0.05):
    return min(1.0, p / warm) if p < warm else max(0.1, 1 - (p - warm) / (1 - warm))


# ---------------------------------------------------------------------------
# Stage 1
# ---------------------------------------------------------------------------
def pos_weights(Y, cap=30.0):
    pos = Y.sum(0).astype(float)
    return torch.tensor(np.clip(np.sqrt((len(Y) - pos) / np.maximum(pos, 1)), 1.0, cap), dtype=torch.float32)


def build_net(exp, make_backbone, device, n_labels, prior, pad_id, log):
    model, d, kind = make_backbone(exp["model"])
    n_backbone = sum(p.numel() for p in model.parameters())
    n_bb, tuner = apply_method(model, kind, exp["method"], dict(exp, d_model=d))
    net = ChunkModel(ChunkEncoder(kind, model, d, tuner, pad_id), d, n_labels, prior).to(device)
    head = list(net.enc.pool.parameters()) + list(net.out.parameters())
    groups = [{"params": head, "lr": exp["lr_head"], "base_lr": exp["lr_head"], "weight_decay": 0.01}]
    if exp["method"] == "partial":
        blocks = list(blocks_of(kind, model))
        norm = list(final_norm(kind, model).parameters())
        groups.append({"params": norm, "lr": exp["lr"], "base_lr": exp["lr"], "weight_decay": 0.0})
        for i, b in enumerate(reversed(blocks[-exp["top_k"]:])):       # i = 0 is the top block
            lr = exp["lr"] * exp.get("lr_decay", 0.85) ** i
            groups.append({"params": list(b.parameters()), "lr": lr, "base_lr": lr, "weight_decay": 0.01})
    elif tuner is not None:
        groups.append({"params": [tuner.prefix], "lr": exp["lr"], "base_lr": exp["lr"], "weight_decay": 0.0})
    else:
        bb = [p for p in model.parameters() if p.requires_grad]
        if bb:
            groups.append({"params": bb, "lr": exp["lr"], "base_lr": exp["lr"], "weight_decay": 0.01})
    return net, kind, n_backbone, n_bb, groups


@torch.no_grad()
def doc_scores_stage1(net, docs, pad_id, device, bs):
    """Stage-1 document logits (log-sum-exp over all kept chunks) [n_docs, L]."""
    net.eval()
    items = [(i, j) for i, d in enumerate(docs) for j in range(len(d["chunks"]))]
    Z = torch.zeros(len(items), net.out.out_features)
    for k in range(0, len(items), bs):
        part = items[k:k + bs]
        ids, mask = pad_tokens([docs[i]["toks"][docs[i]["chunks"][j][0]:docs[i]["chunks"][j][1]] for i, j in part], pad_id, device)
        Z[k:k + len(part)] = net(ids, mask)[0].float().cpu()
    return lse_pool(Z, [i for i, _ in items], len(docs)).numpy()


def train_stage1(exp, data, make_backbone, device, log):
    rng = random.Random(exp["seed"])
    torch.manual_seed(exp["seed"])
    L, pad_id = len(data["labels"]), data["pad_id"]
    by_id = {d["id"]: d for d in data["train"]}
    adapt = [by_id[i] for i in data["split"]["adapt"]]
    val = [by_id[i] for i in data["split"]["agg_val"]][: exp.get("val_docs", 1000)]
    Ya, Yv = label_matrix(adapt, L), label_matrix(val, L)
    net, kind, n_backbone, n_bb, groups = build_net(exp, make_backbone, device, L, prior_logits(Ya), pad_id, log)
    if net.enc.tuner is not None:                   # prefix from real activations (8 long chunks)
        sample = sorted((d["toks"][d["chunks"][0][0]:d["chunks"][0][1]] for d in adapt[:200]), key=len)[-8:]
        net.enc.init_prefix(*pad_tokens(sample, pad_id, device))
    pw = pos_weights(Ya).to(device)
    opt = torch.optim.AdamW(groups)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    mc, accum = exp["micro_chunks"], exp["accum"]
    per_epoch = sum(min(K_TRAIN, len(d["chunks"])) for d in adapt)
    planned = max(1, exp["epochs"] * math.ceil(per_epoch / mc) // accum)
    budget = exp["budget_min"] * 60
    n_head = sum(p.numel() for p in net.enc.pool.parameters()) + sum(p.numel() for p in net.out.parameters())
    pct = 100 * n_bb / n_backbone
    sto = storage_mb(net.enc.m)
    log(f"[{exp['name']}] backbone {exp['model']} {n_backbone / 1e6:.1f}M ({sto} MB stored) | method {exp['method']} | "
        f"trainable {n_bb:,} in the backbone ({pct:.2f}%) + {n_head:,} pooling/head | {L} statutes | adapt docs {len(adapt)} | "
        f"{planned} optimizer steps planned, budget {exp['budget_min']} min")
    t0, step, micro, progress, done = time.time(), 0, 0, 0.0, False
    st = {"best": -1.0, "best_state": None, "hist": [], "run_loss": 0.0, "n_loss": 0}
    eval_points = [0.25, 0.5, 0.75, 1.0]
    params_all = [p for g in groups for p in g["params"]]
    tracked = {id(p) for p in params_all}

    def evaluate():
        ap = macro_ap(Yv, doc_scores_stage1(net, val, pad_id, device, exp["infer_bs"]))
        st["hist"].append({"progress": round(progress, 3), "step": step, "train_loss": round(st["run_loss"] / max(1, st["n_loss"]), 5),
                           "val_macro_ap": ap, "minutes": round((time.time() - t0) / 60, 1)})
        h = st["hist"][-1]
        log(f"[{exp['name']}] progress {progress:.2f} step {step} | train loss {h['train_loss']:.5f} | val macro-AP {ap:.4f} | {h['minutes']} min")
        st["run_loss"], st["n_loss"] = 0.0, 0
        if ap > st["best"]:
            st["best"] = ap
            st["best_state"] = {n: p.detach().cpu().clone() for n, p in net.named_parameters() if id(p) in tracked}

    for epoch in range(exp["epochs"]):
        order = list(range(len(adapt)))
        rng.shuffle(order)
        i = 0
        while i < len(order):
            part, n_ch = [], 0
            while i < len(order):
                d = adapt[order[i]]
                k = min(K_TRAIN, len(d["chunks"]))
                if part and n_ch + k > mc:
                    break
                part.append(d)
                n_ch += k
                i += 1
            net.train()
            progress = min(1.0, max(step / planned, (time.time() - t0) / budget))
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * lr_mult(progress)
            seqs, owners = [], []
            for b, d in enumerate(part):
                for j in rng.sample(range(len(d["chunks"])), min(K_TRAIN, len(d["chunks"]))):
                    s, e = d["chunks"][j][:2]
                    seqs.append(d["toks"][s:e])
                    owners.append(b)
            ids, mask = pad_tokens(seqs, pad_id, device)
            z = lse_pool(net(ids, mask)[0].float(), owners, len(part))
            y = torch.from_numpy(label_matrix(part, L).astype(np.float32)).to(device)
            loss = ml_loss(z, y, "wbce", pw)
            scaler.scale(loss / accum).backward()
            st["run_loss"], st["n_loss"], micro = st["run_loss"] + float(loss.detach()), st["n_loss"] + 1, micro + 1
            if micro % accum:
                continue
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in params_all if p.grad is not None], 1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            step += 1
            progress = min(1.0, max(step / planned, (time.time() - t0) / budget))
            while eval_points and progress >= eval_points[0]:
                eval_points.pop(0)
                evaluate()
            if progress >= 1.0:
                done = True
                break
        if done:
            break
    if eval_points or st["best_state"] is None:
        progress = 1.0
        evaluate()
    net.load_state_dict(st["best_state"], strict=False)
    info = {"n_backbone": n_backbone, "n_trainable_backbone": n_bb, "n_head": n_head, "trainable_pct": round(pct, 3),
            "backbone_storage_mb": sto, "stage1_minutes": round((time.time() - t0) / 60, 1), "stage1_steps": step,
            "stage1_history": st["hist"], "best_val_macro_ap_stage1": st["best"],
            "peak_mem_train_gb": round(torch.cuda.max_memory_allocated(device) / 1e9, 2) if device.type == "cuda" else 0.0}
    if exp["method"] == "qlora":
        info["nf4_relative_weight_error"] = nf4_error(net.enc.m)
    return net, info


def nf4_error(model):
    """Mean relative Frobenius error ||W_nf4 - W|| / ||W|| over the quantised layers."""
    errs = [m._err for m in model.modules() if isinstance(m, QLoRALinear)]
    return round(float(np.mean(errs)), 4) if errs else None


# ---------------------------------------------------------------------------
# Stage 2
# ---------------------------------------------------------------------------
@torch.no_grad()
def encode_docs(net, docs, pad_id, device, bs, view=None):
    """{doc id: (chunk vectors [n, d] fp16, stage-1 chunk logits [n, L] fp16, positions [n, 4], chunk idx)}."""
    net.eval()
    items = []
    for di, d in enumerate(docs):
        js = view(d) if view else range(len(d["chunks"]))
        items += [(di, j) for j in js]
    items.sort(key=lambda t: -(docs[t[0]]["chunks"][t[1]][1] - docs[t[0]]["chunks"][t[1]][0]))
    vec, lg = {}, {}
    for k in range(0, len(items), bs):
        part = items[k:k + bs]
        ids, mask = pad_tokens([docs[di]["toks"][docs[di]["chunks"][j][0]:docs[di]["chunks"][j][1]] for di, j in part], pad_id, device)
        z, v = net(ids, mask)
        v, z = v.half().cpu(), z.half().cpu()
        for q, key in enumerate(part):
            vec[key], lg[key] = v[q], z[q]
    out = {}
    for di, d in enumerate(docs):
        js = list(view(d) if view else range(len(d["chunks"])))
        out[d["id"]] = (torch.stack([vec[(di, j)] for j in js]), torch.stack([lg[(di, j)] for j in js]),
                        np.array([pos_feats(d, j) for j in js], dtype=np.float32), js)
    return out


# ---------------------------------------------------------------------------
# Stage 3
# ---------------------------------------------------------------------------
def stack(enc, ids, docs_by_id, region=None):
    rows = []
    for i in ids:
        V, Z, P, js = enc[i]
        sel = list(range(len(js)))
        if region:
            allowed = set(region_select(docs_by_id[i], region))
            sel = [k for k, j in enumerate(js) if j in allowed] or [0]
        rows.append((V[sel], torch.from_numpy(P[sel])))
    n = max(r[0].shape[0] for r in rows)
    d = rows[0][0].shape[1]
    X = torch.zeros(len(rows), n, d, dtype=torch.float16)
    Pm = torch.zeros(len(rows), n, 4)
    M = torch.zeros(len(rows), n, dtype=torch.bool)
    for b, (v, p) in enumerate(rows):
        X[b, :len(v)], Pm[b, :len(v)], M[b, :len(v)] = v, p, True
    return X, Pm, M


def s1_scores(enc, ids):
    """Stage-1 document logits from cached chunk logits (log-sum-exp pool)."""
    Z = torch.cat([enc[i][1].float() for i in ids])
    owners = [b for b, i in enumerate(ids) for _ in range(len(enc[i][3]))]
    return lse_pool(Z, owners, len(ids)).numpy()


@torch.no_grad()
def reader_predict(rd, X, P, M, device, bs=256):
    rd.eval()
    out = []
    for i in range(0, len(X), bs):
        z, _ = rd(X[i:i + bs].to(device).float(), P[i:i + bs].to(device), M[i:i + bs].to(device))
        out.append(z.float().cpu())
    return torch.cat(out).numpy()


def train_reader(kind, tr, Yt, va, Yv, prior, device, seed, loss="wbce", epochs=40, patience=6, lr=1e-3, bs=64):
    torch.manual_seed(seed)
    rd = Reader(tr[0].shape[-1], Yt.shape[1], kind, prior).to(device)
    opt = torch.optim.AdamW(rd.parameters(), lr=lr, weight_decay=0.01)
    y = torch.from_numpy(Yt.astype(np.float32))
    pw = pos_weights(Yt).to(device)
    best, best_state, bad = -1.0, None, 0
    g = torch.Generator().manual_seed(seed)
    for _ in range(epochs):
        rd.train()
        perm = torch.randperm(len(y), generator=g)
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            z, _ = rd(tr[0][b].to(device).float(), tr[1][b].to(device), tr[2][b].to(device))
            l = ml_loss(z, y[b].to(device), loss, pw)
            opt.zero_grad(set_to_none=True)
            l.backward()
            opt.step()
        ap = macro_ap(Yv, reader_predict(rd, *va, device))
        if ap > best + 1e-5:
            best, bad = ap, 0
            best_state = {k: v.detach().clone() for k, v in rd.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    rd.load_state_dict(best_state)
    return rd, best


def blend(*logit_arrays):
    p = np.mean([1 / (1 + np.exp(-np.clip(a, -30, 30))) for a in logit_arrays], 0)
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def stage3(exp, data, enc, device, log, out_dir, kd_enc=None):
    """Reader study + final document predictions. enc covers agg_train, agg_val, dev, test."""
    seed, L = exp["seed"], len(data["labels"])
    docs = {d["id"]: d for s in ("train", "dev", "test") for d in data[s]}
    tr_ids, va_ids = data["split"]["agg_train"], data["split"]["agg_val"]
    dev_ids, test_ids = [d["id"] for d in data["dev"]], [d["id"] for d in data["test"]]
    Yt, Yv, Yd = (label_matrix([docs[i] for i in ids], L) for ids in (tr_ids, va_ids, dev_ids))
    prior = prior_logits(Yt)
    never = unseen(data)
    rows, cands = [], {}

    def report(name, va_s, dev_s, note):
        thr, cal = calibrate(dev_s, Yd, seed, never=never)
        r = {"reader": name, "note": note, "val_macro_ap": macro_ap(Yv, va_s), "dev_macro_ap": macro_ap(Yd, dev_s),
             "dev_macro_f1": cal["strategy_2fold"][cal["chosen_strategy"]]["macro_f1"],
             "dev_micro_f1": cal["strategy_2fold"][cal["chosen_strategy"]]["micro_f1"]}
        rows.append(r)
        log(f"[{exp['name']}]   {name:30s} val macro-AP {r['val_macro_ap']:.4f} | dev macro-AP {r['dev_macro_ap']:.4f} | "
            f"dev macro-F1 {r['dev_macro_f1']:.4f} (2-fold thresholds) | {note}")
        return r

    log(f"[{exp['name']}] stage 3: readers (trained on {len(tr_ids)} held-out train docs, chosen on {len(va_ids)})")
    full = {s: stack(enc, ids, docs) for s, ids in (("tr", tr_ids), ("va", va_ids), ("dev", dev_ids), ("test", test_ids))}
    s1 = {s: s1_scores(enc, ids) for s, ids in (("va", va_ids), ("dev", dev_ids), ("test", test_ids))}
    report("stage1 log-sum-exp (no reader)", s1["va"], s1["dev"], "whole doc")
    cands["stage1 log-sum-exp (no reader)"] = (s1["va"], s1["dev"], s1["test"])
    readers = {}
    for kind in Reader.KINDS:
        rd, _ = train_reader(kind, full["tr"], Yt, full["va"], Yv, prior, device, seed)
        readers[kind] = rd
        sc = tuple(reader_predict(rd, *full[s], device) for s in ("va", "dev", "test"))
        cands[kind] = sc
        report(kind, sc[0], sc[1], "whole doc")
    best_reader = max(Reader.KINDS, key=lambda k: rows[[r["reader"] for r in rows].index(k)]["val_macro_ap"])
    bl = tuple(blend(cands[best_reader][k], cands["stage1 log-sum-exp (no reader)"][k]) for k in range(3))
    cands[f"blend: {best_reader} + stage1"] = bl
    report(f"blend: {best_reader} + stage1", bl[0], bl[1], "whole doc")
    # loss study (label-wise attention reader)
    for loss in LOSSES:
        if loss == "wbce":
            continue
        rd, _ = train_reader("lwan", full["tr"], Yt, full["va"], Yv, prior, device, seed, loss=loss)
        report(f"lwan, loss {loss}", reader_predict(rd, *full["va"], device), reader_predict(rd, *full["dev"], device), "loss study")
    # section study
    for region in ("first", "begin", "middle", "end"):
        sub = {s: stack(enc, ids, docs, region=region) for s, ids in (("tr", tr_ids), ("va", va_ids), ("dev", dev_ids))}
        rd, _ = train_reader("lwan", sub["tr"], Yt, sub["va"], Yv, prior, device, seed)
        report(f"lwan on {region} {'chunk' if region == 'first' else 'third'} only", reader_predict(rd, *sub["va"], device),
               reader_predict(rd, *sub["dev"], device), "section study")
    whole = [r for r in rows if r["note"] == "whole doc"]
    chosen = max(whole, key=lambda r: r["val_macro_ap"])["reader"]
    _, dev_s, test_s = cands[chosen]
    thr, cal = calibrate(dev_s, Yd, seed, never=never)
    Pd, Pt = apply_thresholds(dev_s, thr), apply_thresholds(test_s, thr)
    rep = multilabel_report(Yd, Pd)
    log(f"[{exp['name']}] chosen: {chosen} | thresholds: {cal['chosen_strategy']} (2-fold dev: " +
        ", ".join(f"{k} {v['macro_f1']:.4f}" for k, v in cal["strategy_2fold"].items()) +
        f") | dev macro-F1 {rep['macro_f1']:.4f} micro-F1 {rep['micro_f1']:.4f} subset acc {rep['subset_accuracy']:.4f}")
    with open(Path(out_dir) / "readers.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)
    res = {"readers": rows, "chosen_reader": chosen, "calibration": cal, "dev_report": rep,
           "dev_report_2fold": cal["strategy_2fold"][cal["chosen_strategy"]], "dev_macro_ap": macro_ap(Yd, dev_s),
           "threshold": thr.tolist(), "dev_scores": dev_s, "test_scores": test_s, "test_pred": Pt,
           "analysis": analysis(Yd, Pd, dev_s, thr, data, [docs[i] for i in dev_ids], out_dir)}
    if kd_enc is not None:                          # teacher targets on every train document (KD view)
        tr_all = [d["id"] for d in data["train"]]
        Xk = stack(kd_enc, tr_all, docs)
        if chosen.startswith("stage1"):
            lg = s1_scores(kd_enc, tr_all)
        elif chosen.startswith("blend"):
            lg = blend(reader_predict(readers[best_reader], *Xk, device), s1_scores(kd_enc, tr_all))
        else:
            lg = reader_predict(readers[chosen], *Xk, device)
        torch.save({"ids": tr_all, "logits": torch.from_numpy(lg).half(), "labels": data["labels"]}, Path(out_dir) / "kd_targets.pt")
        log(f"[{exp['name']}] wrote distillation targets for {len(tr_all)} train documents")
    return res


def analysis(Yd, Pd, Sd, thr, data, dev_docs, out_dir):
    """Per-statute table + rare-statute buckets + documents with many statutes + length + most
    missed / most over-predicted statutes + (missed, wrongly predicted instead) pairs."""
    labels = data["labels"]
    L = len(labels)
    train_freq = label_matrix(data["train"], L).sum(0)
    f1, tp, fp, fn = per_label_f1(Yd, Pd)
    present = Yd.any(0)
    with open(Path(out_dir) / "per_statute.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["statute", "train_docs", "dev_docs", "dev_ap", "dev_f1", "tp", "fp", "fn", "threshold"])
        for l in np.argsort(-train_freq):
            w.writerow([labels[l], int(train_freq[l]), int(Yd[:, l].sum()), round(average_precision(Yd[:, l], Sd[:, l]), 4) if present[l] else "",
                        round(float(f1[l]), 4), int(tp[l]), int(fp[l]), int(fn[l]), round(float(thr[l]), 4) if np.isfinite(thr[l]) else "never"])
    order = np.argsort(-train_freq)
    thirds = np.array_split(order, 3)
    buckets = []
    for name, idx in zip(("frequent third", "middle third", "rare third"), thirds):
        idx = [l for l in idx if present[l]]
        if idx:
            buckets.append({"statutes": name, "n_statutes": len(idx), "train_docs_from": int(train_freq[idx].min()),
                            "train_docs_to": int(train_freq[idx].max()), "macro_f1": round(float(f1[idx].mean()), 4)})
    card = Yd.sum(1)
    ef1 = example_f1(Yd, Pd)
    by_card = []
    for lo, hi, name in ((1, 1, "1"), (2, 2, "2"), (3, 3, "3"), (4, 10 ** 6, "4+")):
        m = (card >= lo) & (card <= hi)
        if m.any():
            by_card.append({"gold_statutes": name, "docs": int(m.sum()), "example_f1": round(float(ef1[m].mean()), 4),
                            "avg_predicted": round(float(Pd[m].sum(1).mean()), 2)})
    ntok = np.array([d["n_tok"] for d in dev_docs])
    qs = np.quantile(ntok, [0.25, 0.5, 0.75])
    by_len = []
    for k, (lo, hi) in enumerate(((0, qs[0]), (qs[0], qs[1]), (qs[1], qs[2]), (qs[2], np.inf))):
        m = (ntok >= lo) & (ntok < hi) if k < 3 else (ntok >= lo)
        if m.any():
            by_len.append({"length_quartile": k + 1, "tokens_from": int(lo), "docs": int(m.sum()), "example_f1": round(float(ef1[m].mean()), 4)})
    pairs = Counter()
    for i in range(len(Yd)):
        miss, extra = np.where(Yd[i] & ~Pd[i])[0], np.where(~Yd[i] & Pd[i])[0]
        for a in miss:
            for b in extra:
                pairs[(labels[a], labels[b])] += 1
    return {"by_statute_frequency": buckets, "by_gold_count": by_card, "by_length": by_len,
            "most_missed": [(labels[l], int(fn[l])) for l in np.argsort(-fn)[:10] if fn[l]],
            "most_over_predicted": [(labels[l], int(fp[l])) for l in np.argsort(-fp)[:10] if fp[l]],
            "missed_vs_predicted_instead": [(a, b, c) for (a, b), c in pairs.most_common(10)],
            "statutes_never_predicted": int(((Pd.sum(0) == 0) & present).sum()), "statutes_in_dev": int(present.sum())}


# ---------------------------------------------------------------------------
# Distillation student
# ---------------------------------------------------------------------------
class Student(nn.Module):
    """Small encoder + label-wise attention reader over the KD view (<= KD_CHUNKS chunks), end to end."""

    def __init__(self, encoder, d, n_labels, prior):
        super().__init__()
        self.enc = encoder
        self.reader = Reader(d, n_labels, "lwan", prior)

    def forward(self, docs, pad_id, device):
        seqs, owners, P = [], [], []
        for b, d in enumerate(docs):
            for j in kd_view(d):
                s, e = d["chunks"][j][:2]
                seqs.append(d["toks"][s:e])
                owners.append(b)
                P.append(pos_feats(d, j))
        ids, mask = pad_tokens(seqs, pad_id, device)
        v = self.enc(ids, mask)
        n = max(Counter(owners).values())
        X = torch.zeros(len(docs), n, v.shape[-1], device=device)
        Pm = torch.zeros(len(docs), n, 4, device=device)
        M = torch.zeros(len(docs), n, dtype=torch.bool, device=device)
        fill = [0] * len(docs)
        Pt = torch.tensor(P, device=device)
        for k, b in enumerate(owners):
            X[b, fill[b]], Pm[b, fill[b]], M[b, fill[b]] = v[k], Pt[k], True
            fill[b] += 1
        return self.reader(X, Pm, M)


@torch.no_grad()
def student_scores(net, docs, pad_id, device, bs):
    net.eval()
    out = []
    for i in range(0, len(docs), bs):
        out.append(net(docs[i:i + bs], pad_id, device)[0].float().cpu())
    return torch.cat(out).numpy()


def run_student(exp, data, make_backbone, device, log, out_dir, teacher_files):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    rng = random.Random(exp["seed"])
    torch.manual_seed(exp["seed"])
    L, pad_id = len(data["labels"]), data["pad_id"]
    model, d, kind = make_backbone(exp["model"])
    for p in model.parameters():
        p.requires_grad_(True)
    n_backbone = sum(p.numel() for p in model.parameters())
    by_id = {x["id"]: x for x in data["train"]}
    val_ids = set(data["split"]["agg_val"])
    train = [x for x in data["train"] if x["id"] not in val_ids]
    val = [by_id[i] for i in data["split"]["agg_val"]][: exp.get("val_docs", 1000)]
    Ytr, Yv = label_matrix(train, L), label_matrix(val, L)
    net = Student(ChunkEncoder(kind, model, d), d, L, prior_logits(Ytr)).to(device)
    T, alpha = exp.get("kd_T", 2.0), exp.get("kd_alpha", 0.0)
    teach = None
    if teacher_files and alpha > 0:
        tl = []
        for f in teacher_files:
            t = torch.load(f, weights_only=False)
            assert t["labels"] == data["labels"], "teacher statute order differs"
            pos = {i: k for k, i in enumerate(t["ids"])}
            tl.append(t["logits"].float()[[pos[x["id"]] for x in data["train"]]])
        teach = dict(zip([x["id"] for x in data["train"]], torch.stack(tl).mean(0)))   # mean teacher logit
    pw = pos_weights(Ytr).to(device)
    groups = [{"params": list(net.enc.m.parameters()), "lr": exp["lr"], "base_lr": exp["lr"], "weight_decay": 0.01},
              {"params": list(net.enc.pool.parameters()) + list(net.reader.parameters()), "lr": exp["lr_head"],
               "base_lr": exp["lr_head"], "weight_decay": 0.01}]
    opt = torch.optim.AdamW(groups)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    mb, accum = exp["micro_docs"], exp["accum"]
    planned = max(1, exp["epochs"] * math.ceil(len(train) / mb) // accum)
    budget = exp["budget_min"] * 60
    n_tr = sum(p.numel() for p in net.parameters() if p.requires_grad)
    sto = storage_mb(model)
    log(f"[{exp['name']}] student {exp['model']} {n_backbone / 1e6:.1f}M ({sto} MB) = Phase-1 D encoder layers "
        f"{exp.get('student_layers')} | reads <= {KD_CHUNKS} chunks | teachers: {exp.get('teacher_names') or 'none (gold labels only)'} "
        f"| alpha {alpha} T {T} | {len(train)} docs, {planned} steps planned, budget {exp['budget_min']} min")
    t0, step, micro, progress, done = time.time(), 0, 0, 0.0, False
    st = {"best": -1.0, "best_state": None, "hist": [], "run_loss": 0.0, "n_loss": 0}
    eval_points = [0.25, 0.5, 0.75, 1.0]

    def evaluate():
        ap = macro_ap(Yv, student_scores(net, val, pad_id, device, exp["infer_docs"]))
        st["hist"].append({"progress": round(progress, 3), "step": step, "train_loss": round(st["run_loss"] / max(1, st["n_loss"]), 5),
                           "val_macro_ap": ap, "minutes": round((time.time() - t0) / 60, 1)})
        h = st["hist"][-1]
        log(f"[{exp['name']}] progress {progress:.2f} step {step} | loss {h['train_loss']:.5f} | val macro-AP {ap:.4f} | {h['minutes']} min")
        st["run_loss"], st["n_loss"] = 0.0, 0
        if ap > st["best"]:
            st["best"] = ap
            st["best_state"] = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}

    for epoch in range(exp["epochs"]):
        order = list(range(len(train)))
        rng.shuffle(order)
        for i in range(0, len(order), mb):
            net.train()
            progress = min(1.0, max(step / planned, (time.time() - t0) / budget))
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * lr_mult(progress)
            part = [train[k] for k in order[i:i + mb]]
            z = net(part, pad_id, device)[0].float()
            y = torch.from_numpy(label_matrix(part, L).astype(np.float32)).to(device)
            loss = ml_loss(z, y, "wbce", pw)
            if teach is not None:
                pt = torch.sigmoid(torch.stack([teach[x["id"]] for x in part]).to(device) / T)
                kd = F.binary_cross_entropy_with_logits(z / T, pt) * T * T
                loss = (1 - alpha) * loss + alpha * kd
            scaler.scale(loss / accum).backward()
            st["run_loss"], st["n_loss"], micro = st["run_loss"] + float(loss.detach()), st["n_loss"] + 1, micro + 1
            if micro % accum:
                continue
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            step += 1
            progress = min(1.0, max(step / planned, (time.time() - t0) / budget))
            while eval_points and progress >= eval_points[0]:
                eval_points.pop(0)
                evaluate()
            if progress >= 1.0:
                done = True
                break
        if done:
            break
    if eval_points or st["best_state"] is None:
        progress = 1.0
        evaluate()
    net.load_state_dict(st["best_state"])
    train_min = round((time.time() - t0) / 60, 1)
    peak_train = round(torch.cuda.max_memory_allocated(device) / 1e9, 2) if device.type == "cuda" else 0.0
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    t1 = time.time()
    dev_s = student_scores(net, data["dev"], pad_id, device, exp["infer_docs"])
    infer_s = time.time() - t1
    test_s = student_scores(net, data["test"], pad_id, device, exp["infer_docs"])
    Yd = label_matrix(data["dev"], L)
    thr, cal = calibrate(dev_s, Yd, exp["seed"], never=unseen(data))
    Pd = apply_thresholds(dev_s, thr)
    rep = multilabel_report(Yd, Pd)
    log(f"[{exp['name']}] thresholds: {cal['chosen_strategy']} (2-fold dev: " +
        ", ".join(f"{k} {v['macro_f1']:.4f}" for k, v in cal["strategy_2fold"].items()) + ")")
    info = {"exp": exp, "n_backbone": n_backbone, "n_trainable_backbone": n_backbone, "n_head": n_tr - n_backbone,
            "trainable_pct": 100.0, "backbone_storage_mb": sto, "train_minutes": train_min, "history": st["hist"],
            "best_val_macro_ap": st["best"], "peak_mem_train_gb": peak_train,
            "peak_mem_infer_gb": round(torch.cuda.max_memory_allocated(device) / 1e9, 2) if device.type == "cuda" else 0.0,
            "infer_docs_per_s": round(len(data["dev"]) / max(1e-6, infer_s), 2), "calibration": cal, "dev_report": rep,
            "dev_report_2fold": cal["strategy_2fold"][cal["chosen_strategy"]],
            "dev_macro_ap": macro_ap(Yd, dev_s), "chosen_reader": f"student: label-wise attention over <= {KD_CHUNKS} chunks",
            "threshold": thr.tolist(), "analysis": analysis(Yd, Pd, dev_s, thr, data, data["dev"], out_dir)}
    write_outputs(out_dir, info, data, dev_s, test_s, apply_thresholds(test_s, thr), log)
    return info


# ---------------------------------------------------------------------------
# One standard experiment, outputs
# ---------------------------------------------------------------------------
def run_standard(exp, data, make_backbone, device, log, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    net, info = train_stage1(exp, data, make_backbone, device, log)
    by_id = {d["id"]: d for d in data["train"]}
    agg = [by_id[i] for i in data["split"]["agg_train"] + data["split"]["agg_val"]]
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    t0 = time.time()
    enc = encode_docs(net, data["dev"], data["pad_id"], device, exp["infer_bs"])
    dev_s = time.time() - t0
    info["infer_docs_per_s"] = round(len(data["dev"]) / max(1e-6, dev_s), 2)
    info["peak_mem_infer_gb"] = round(torch.cuda.max_memory_allocated(device) / 1e9, 2) if device.type == "cuda" else 0.0
    enc.update(encode_docs(net, data["test"], data["pad_id"], device, exp["infer_bs"]))
    enc.update(encode_docs(net, agg, data["pad_id"], device, exp["infer_bs"]))
    kd_enc = None
    if exp.get("kd_teacher"):
        kd_enc = encode_docs(net, data["train"], data["pad_id"], device, exp["infer_bs"], view=kd_view)
    info["encode_minutes"] = round((time.time() - t0) / 60, 1)
    log(f"[{exp['name']}] encoded chunks in {info['encode_minutes']} min ({info['infer_docs_per_s']} dev docs/s, all chunks)")
    del net
    if device.type == "cuda":
        torch.cuda.empty_cache()
    res = stage3(exp, data, enc, device, log, out_dir, kd_enc)
    dev_scores, test_scores, test_pred = res.pop("dev_scores"), res.pop("test_scores"), res.pop("test_pred")
    info.update(res)
    info["exp"] = exp
    write_outputs(out_dir, info, data, dev_scores, test_scores, test_pred, log)
    return info


def submission_rows(test_docs, P, labels):
    rows, seen = [], set()
    for d, p in zip(test_docs, P):
        assert d["id"] not in seen, f"duplicate test id {d['id']}"
        seen.add(d["id"])
        pred = sorted({int(labels[j]) for j in np.where(p)[0]})
        assert pred, f"empty prediction for {d['id']}"
        rows.append({"id": d["raw_id"], "pred": pred})
    return rows


def write_submission(out_dir, rows):
    out_dir = Path(out_dir)
    with open(out_dir / "prediction.json", "w") as f:
        json.dump(rows, f)
    with zipfile.ZipFile(out_dir / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(out_dir / "prediction.json", arcname="prediction.json")


def write_outputs(out_dir, info, data, dev_scores, test_scores, test_pred, log):
    out_dir = Path(out_dir)
    rows = submission_rows(data["test"], test_pred, data["labels"])
    write_submission(out_dir, rows)
    np.savez_compressed(out_dir / "scores.npz", dev=np.asarray(dev_scores, np.float32), test=np.asarray(test_scores, np.float32),
                        dev_ids=np.array([d["id"] for d in data["dev"]]), test_ids=np.array([d["id"] for d in data["test"]]),
                        labels=np.array(data["labels"]))
    with open(out_dir / "metrics.json", "w") as f:
        json.dump(info, f, indent=1, default=str)
    r = info["dev_report"]
    log(f"[{info['exp']['name']}] DONE dev macro-F1 {info['dev_report_2fold']['macro_f1']:.4f} with 2-fold thresholds (honest), "
        f"{r['macro_f1']:.4f} with thresholds fitted on all dev (P {r['macro_precision']:.4f} R {r['macro_recall']:.4f}) | "
        f"micro-F1 {r['micro_f1']:.4f} | subset acc {r['subset_accuracy']:.4f} | {r['avg_predicted_per_doc']} statutes/doc | "
        f"-> {out_dir / 'submission.zip'}")


# ---------------------------------------------------------------------------
# Self-tests (tiny models, CPU, seconds)
# ---------------------------------------------------------------------------
def _vec(enc, seqs, device):
    with torch.no_grad():
        return enc(*pad_tokens(seqs, 0, device))


def self_test(make_tiny, device, kinds=("D", "C", "A", "B")):
    torch.manual_seed(0)
    a, b = np.array([5, 9, 7, 11, 6, 8, 3, 2, 4, 9]), np.array([4, 10, 12, 3])
    base_exp = {"r": 4, "alpha": 8, "bottleneck": 8, "top_k": 1, "n_prefix": 2}
    for kind in kinds:
        m0, d = make_tiny(kind)
        ref = ChunkEncoder(kind, m0, d).to(device).eval()
        before = _vec(ref, [a, b], device)
        assert (_vec(ref, [b], device)[0] - before[1]).abs().max() < 1e-4, f"{kind}: padding changes the chunk vector"
        for method in ("lorma", "adapter", "qlora", "partial", "full", "prefix"):
            m, d = make_tiny(kind)
            m.load_state_dict(m0.state_dict())
            n_tr, tuner = apply_method(m, kind, method, dict(base_exp, d_model=d))
            enc = ChunkEncoder(kind, m, d, tuner).to(device).eval()
            enc.pool.load_state_dict(ref.pool.state_dict())
            after = _vec(enc, [a, b], device)
            if method in ("lorma", "adapter", "partial", "full"):
                assert (before - after).abs().max() < 1e-4, f"{kind}/{method} does not start as the Phase-1 model"
            elif method == "qlora":
                cos = F.cosine_similarity(before, after, -1).min()
                assert cos > 0.95, f"{kind}/qlora: 4-bit model too far from the Phase-1 model (cos {cos:.3f})"
                for q in [x for x in m.modules() if isinstance(x, QLoRALinear)][:2]:
                    x = torch.randn(3, q.in_features, device=device)
                    assert (q(x) - F.linear(x, q.weight(), q.bias)).abs().max() < 1e-5, "QLoRA must start as the 4-bit layer"
            else:                                   # prefix: padding-safe, only the prefix learns, hooks removable
                assert (_vec(enc, [b], device)[0] - after[1]).abs().max() < 1e-4, f"{kind}/prefix: padding changes the vector"
                enc.train()
                enc(*pad_tokens([a, b], 0, device)).sum().backward()
                assert tuner.prefix.grad is not None and tuner.prefix.grad.abs().sum() > 0, f"{kind}/prefix gets no gradient"
                assert all(p.grad is None for p in m.parameters()), f"{kind}/prefix: the backbone must stay frozen"
                tuner.remove()
                plain = ChunkEncoder(kind, m, d).to(device).eval()
                plain.pool.load_state_dict(ref.pool.state_dict())
                assert (_vec(plain, [a, b], device) - before).abs().max() < 1e-4, f"{kind}: prefix hooks did not detach"
            if method == "partial":
                top = list(blocks_of(kind, m))[-1]
                assert all(p.requires_grad for p in top.parameters())
                assert not any(p.requires_grad for p in list(blocks_of(kind, m))[0].parameters())
            if method in ("lorma", "qlora"):
                assert n_tr > 0 and all(("A" in n.split(".")[-1] or "B" in n.split(".")[-1]) for n, p in m.named_parameters() if p.requires_grad)
    # stage-1 model (label-wise token attention head) is padding-safe
    m, d = make_tiny(kinds[0])
    cm = ChunkModel(ChunkEncoder(kinds[0], m, d), d, 5, torch.zeros(5)).to(device).eval()
    with torch.no_grad():
        z2, v2 = cm(*pad_tokens([a, b], 0, device))
        z1, v1 = cm(*pad_tokens([b], 0, device))
    assert (z2[1] - z1[0]).abs().max() < 1e-4 and (v2[1] - v1[0]).abs().max() < 1e-4, "stage-1 head depends on padding"
    # NF4 + double quantisation
    w = torch.randn(96, 80)
    q = quantize_nf4(w, 64, 4)
    w2 = dequantize_nf4(q["codes"], q["qabs"], q["s2"], q["c"], torch.tensor(NF4_VALUES), q["n"], q["nb"], q["shape"], 64, 4)
    rel = float((w2 - w).norm() / w.norm())
    assert rel < 0.12 and q["codes"].dtype == torch.uint8 and q["codes"].numel() == w.numel() // 2, rel
    # log-sum-exp pooling
    z = torch.tensor([[1.0, -2.0], [3.0, 0.0], [0.5, 0.5]])
    p = lse_pool(z, [0, 0, 1], 2)
    assert torch.allclose(p[1], z[2]) and (p[0] <= z[:2].max(0).values + 1e-6).all() and (p[0] >= z[:2].mean(0) - 1e-6).all()
    # readers are padding-safe
    L = 5
    X, P = torch.randn(2, 5, 16), torch.rand(2, 5, 4)
    M = torch.tensor([[1, 1, 1, 0, 0], [1, 1, 1, 1, 1]], dtype=torch.bool)
    for kind in Reader.KINDS:
        rd = Reader(16, L, kind, torch.zeros(L), h=8).eval()
        z, _ = rd(X, P, M)
        z1, _ = rd(X[:1, :3], P[:1, :3], M[:1, :3])
        assert z.shape == (2, L) and (z[0] - z1[0]).abs().max() < 1e-4, f"reader {kind} depends on padding"
    # losses
    zz, yy = torch.randn(4, L, requires_grad=True), (torch.rand(4, L) > 0.7).float()
    for kind in LOSSES:
        l = ml_loss(zz, yy, kind, torch.full((L,), 2.0))
        l.backward()
        assert torch.isfinite(l) and zz.grad is not None
    # student
    ks = "D" if "D" in kinds else kinds[0]
    m, d = make_tiny(ks)
    st = Student(ChunkEncoder(ks, m, d), d, L, torch.zeros(L)).to(device)
    fake = [{"toks": (np.arange(1, 60) % 30 + 1).astype(np.uint16), "chunks": [(0, 20, 0, .3), (15, 35, .25, .6), (30, 50, .5, .8),
                                                                              (40, 59, .7, .9), (45, 59, .8, 1)]},
            {"toks": np.arange(1, 9, dtype=np.uint16), "chunks": [(0, 8, 0, 1)]}]
    lg, w_ = st(fake, 0, device)
    lg.sum().backward()
    assert lg.shape == (2, L) and w_.shape[1] == KD_CHUNKS
    # metrics, thresholds
    rng = np.random.RandomState(0)
    Y = rng.rand(300, 6) < np.array([0.4, 0.2, 0.1, 0.05, 0.02, 0.0])
    S = Y * rng.randn(300, 6) * 0.5 + Y * 1.5 + rng.randn(300, 6)
    Pp = S > 0.5
    rep = multilabel_report(Y, Pp)
    try:
        from sklearn.metrics import average_precision_score, f1_score
        cols = Y.any(0)
        assert abs(rep["macro_f1"] - f1_score(Y[:, cols], Pp[:, cols], average="macro", zero_division=0)) < 1e-4
        assert abs(rep["micro_f1"] - f1_score(Y[:, cols], Pp[:, cols], average="micro", zero_division=0)) < 1e-4
        assert abs(average_precision(Y[:, 0], S[:, 0]) - average_precision_score(Y[:, 0], S[:, 0])) < 1e-9
    except ImportError:
        pass
    thr = best_label_thresholds(S, Y)
    for l in range(5):
        brute = max(per_label_f1(Y[:, [l]], S[:, [l]] > t)[0][0] for t in np.r_[S[:, l], -1e9])
        assert abs(per_label_f1(Y[:, [l]], S[:, [l]] > thr[l])[0][0] - brute) < 1e-9, "per-statute threshold is not optimal"
    assert np.isnan(thr[5])
    Pz = apply_thresholds(S, np.full(6, 1e9))
    assert (Pz.sum(1) == 1).all(), "every document must get at least one statute"
    t_, cal = calibrate(S, Y)
    assert cal["chosen_strategy"] in THRESHOLD_STRATEGIES and np.isfinite(t_).all()
    nv = np.array([False, True, False, False, False, False])
    t_, cal = calibrate(S, Y, never=nv)
    assert np.isinf(t_[1]) and not apply_thresholds(S, t_)[:, 1].any(), "unseen statutes must never be predicted"
    print("lsi_lib self-test passed: LoRMA / adapters / partial / full start as the Phase-1 model, QLoRA starts as its 4-bit "
          f"copy, prefix tuning trains only the prefix and is padding-safe ({', '.join(kinds)}); NF4 round trip "
          f"{rel:.3f} relative error; log-sum-exp pooling; all {len(Reader.KINDS)} readers padding-safe; 4 losses; student "
          "trains; macro/micro-F1 and AP match scikit-learn; per-statute thresholds optimal; top-1 fallback; unseen statutes never predicted.")

In [ ]:
%%writefile /kaggle/working/lsi_train.py
"""lsi_train.py -- one Phase-5 experiment: python lsi_train.py <config.json> <experiment> <gpu index or -1>"""
import json
import sys
from pathlib import Path

import torch

cfg = json.load(open(sys.argv[1]))
name, gpu = sys.argv[2], int(sys.argv[3])
sys.path.insert(0, cfg["work_dir"])
import lsi_lib as LSI  # noqa: E402

device = torch.device(f"cuda:{gpu}") if gpu >= 0 and torch.cuda.is_available() else torch.device("cpu")
if device.type == "cuda":
    torch.cuda.set_device(device)
exp = dict(cfg["common"], **cfg["experiments"][name], name=name)
TOKENIZER = {"A": "T1", "B": "T1", "C": "T2", "D": "T2", "Ds": "T2"}
data = torch.load(cfg["data"][TOKENIZER[exp["model"]]], weights_only=False)


def make_backbone(kind):
    """Phase-1 model from its checkpoint -> (module, d_model, kind used by the encoder)."""
    if kind in ("D", "Ds"):
        import legal_d
        ck = torch.load(cfg["ckpt"]["D"], map_location="cpu", weights_only=False)
        m = legal_d.build_model(ck["config"])
        m.load_state_dict(ck["model_state"])
        m.dec_blocks = torch.nn.ModuleList()          # encoder only: the decoder is not used for classification
        if kind == "Ds":                              # distillation student: every other encoder layer (DistilBERT-style)
            keep = [i for i in exp["student_layers"] if i < len(m.enc_blocks)]
            m.enc_blocks = torch.nn.ModuleList([m.enc_blocks[i] for i in keep])
            exp["student_layers"] = keep
        m.grad_ckpt = exp.get("grad_ckpt", True)
        return m, ck["config"]["d_model"], kind
    import importlib
    mod = importlib.import_module({"A": "a_model", "B": "b_model", "C": "c_model"}[kind])
    ck = torch.load(cfg["ckpt"][kind], map_location="cpu", weights_only=False)
    m = mod.build_model(ck["config"])
    m.load_state_dict(ck["model_state"])
    mod.USE_GRAD_CHECKPOINTING = exp.get("grad_ckpt", True)
    return m, ck["config"]["d_model"], kind


def log(*a):
    print(*a, flush=True)


out_dir = Path(cfg["out_dir"]) / name
if exp.get("type") == "student":
    files = [Path(cfg["out_dir"]) / t / "kd_targets.pt" for t in exp.get("teachers", [])]
    for f in files:
        assert f.exists(), f"missing teacher targets {f}"
    LSI.run_student(exp, data, make_backbone, device, log, out_dir, files)
else:
    LSI.run_standard(exp, data, make_backbone, device, log, out_dir)

In [ ]:
%%writefile /kaggle/working/lsi_tfidf.py
"""lsi_tfidf.py -- reference WITHOUT any pretrained model (dev only, never submitted):
hashed word 1-2-gram TF-IDF + one logistic regression per statute, same thresholds as the runs.
python lsi_tfidf.py <clean_docs.pt> <out.json> <work_dir>"""
import json
import sys
import time

import numpy as np
import torch
from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

sys.path.insert(0, sys.argv[3])
import lsi_lib as LSI  # noqa: E402

t0 = time.time()
C = torch.load(sys.argv[1], weights_only=False)
labels, train, dev = C["labels"], C["train"], C["dev"]
L = len(labels)
Ytr, Yd = LSI.label_matrix(train, L), LSI.label_matrix(dev, L)
cut = lambda t: " ".join(t.split()[:3000])
vec = HashingVectorizer(n_features=2 ** 19, ngram_range=(1, 2), alternate_sign=False, norm=None, lowercase=True)
tf = TfidfTransformer(sublinear_tf=True)
Xtr = tf.fit_transform(vec.transform(cut(d["text"]) for d in train))
Xd = tf.transform(vec.transform(cut(d["text"]) for d in dev))
print(f"[TF-IDF] features ready ({time.time() - t0:.0f}s), {L} statutes", flush=True)
cols = np.where((Ytr.sum(0) >= 2) & (Ytr.sum(0) <= len(Ytr) - 2))[0]          # learnable statutes
clf = OneVsRestClassifier(LogisticRegression(C=4.0, class_weight="balanced", max_iter=2000, solver="liblinear"), n_jobs=2)
clf.fit(Xtr, Ytr[:, cols])
S = np.full((len(dev), L), -20.0)
S[:, cols] = np.asarray(clf.decision_function(Xd)).reshape(len(dev), -1)
never = np.ones(L, bool)
never[cols] = False
thr, cal = LSI.calibrate(S, Yd, never=never)
rep = LSI.multilabel_report(Yd, LSI.apply_thresholds(S, thr))
res = {"dev_report": rep, "dev_report_2fold": cal["strategy_2fold"][cal["chosen_strategy"]], "dev_macro_ap": LSI.macro_ap(Yd, S), "calibration": cal, "minutes": round((time.time() - t0) / 60, 1)}
print(f"[TF-IDF] dev macro-F1 {res['dev_report_2fold']['macro_f1']:.4f} (2-fold thresholds), {rep['macro_f1']:.4f} (fit) | micro-F1 {rep['micro_f1']:.4f} | macro-AP {res['dev_macro_ap']:.4f} "
      f"| thresholds {cal['chosen_strategy']} ({res['minutes']} min)", flush=True)
json.dump(res, open(sys.argv[2], "w"), indent=1)

## 3. Data: download, clean, analyse

In [ ]:
sys.path.insert(0, str(WORK_DIR))
import lsi_lib as LSI  # noqa: E402
from huggingface_hub import hf_hub_download  # noqa: E402
from kaggle_secrets import UserSecretsClient  # noqa: E402

HF_TOKEN = UserSecretsClient().get_secret("HF")
REPO = "Exploration-Lab/CS-781-Capstone"


def hf_file(*names):
    # first of the candidate paths that exists in the repo (the capstone doc names the first one)
    last = None
    for n in names:
        try:
            return hf_hub_download(REPO, n, repo_type="dataset", token=HF_TOKEN)
        except Exception as e:      # noqa: BLE001
            last = e
    try:
        from huggingface_hub import list_repo_files
        have = [f for f in list_repo_files(REPO, repo_type="dataset", token=HF_TOKEN) if "5" in f.split("/")[0]]
    except Exception:               # noqa: BLE001
        have = "?"
    raise FileNotFoundError(f"none of {names} in {REPO} ({last}). Phase-5 files in the repo: {have}")


t0 = time.time()
RAW = {"train": hf_file("Phase_5_LSI/Train/train.json", "Phase_5_LSI/Train/train.jsonl"),
       "dev": hf_file("Phase_5_LSI/Train/dev.json", "Phase_5_LSI/Train/dev.jsonl", "Phase_5_LSI/Dev/dev.json", "Phase_5_LSI/Dev/dev.jsonl"),
       "test": hf_file("Phase_5_LSI/Test/test.json", "Phase_5_LSI/Test/test.jsonl")}
docs = {s: LSI.parse_docs(LSI.load_json_any(p)) for s, p in RAW.items()}
print(f"downloaded + parsed in {time.time() - t0:.0f}s:", {s: len(v) for s, v in docs.items()})
for s in ("train", "dev"):
    miss = sum(d["labels"] is None for d in docs[s])
    assert miss == 0, f"{s}: {miss} documents have no 'labels' field"
for s in docs:
    empty = sum(not d["text"] for d in docs[s])
    assert empty <= 0.05 * len(docs[s]), f"{s}: {empty} documents have no text -- check the field names: {list(LSI.load_json_any(RAW[s])[0])}"
print("example:", {k: (v[:150] + "...") if isinstance(v, str) else v for k, v in docs["train"][0].items()})
docs, CLEAN = LSI.clean_splits(docs)
print("cleaning:", CLEAN)
LABELS = LSI.label_space(docs["train"], docs["dev"])
LIDX = {l: i for i, l in enumerate(LABELS)}
L = len(LABELS)
for s in ("train", "dev"):
    for d in docs[s]:
        d["y"] = [LIDX[l] for l in d["labels"]]
Ytr, Ydv = LSI.label_matrix(docs["train"], L), LSI.label_matrix(docs["dev"], L)

# ---- data analysis ---------------------------------------------------------
REPORT = {"cleaning": CLEAN, "n_statutes": L, "statute_ids": LABELS}
ftr, fdv = Ytr.sum(0), Ydv.sum(0)
print(f"\n{L} statutes (ids {LABELS[0]}..{LABELS[-1]}); only in dev: {[LABELS[l] for l in np.where((ftr == 0) & (fdv > 0))[0]]}; "
      f"not in dev: {int((fdv == 0).sum())}")
order = np.argsort(-ftr)
print("most frequent statutes (id: train docs / dev docs):", [(LABELS[l], int(ftr[l]), int(fdv[l])) for l in order[:10]])
print("rarest statutes:", [(LABELS[l], int(ftr[l]), int(fdv[l])) for l in order[-10:]])
for k in (10, 50, 100, 500):
    print(f"  statutes with < {k} train docs: {int((ftr < k).sum())}")
REPORT["train_docs_per_statute"] = {str(LABELS[l]): int(ftr[l]) for l in order}
REPORT["dev_docs_per_statute"] = {str(LABELS[l]): int(fdv[l]) for l in order}
for s, Y in (("train", Ytr), ("dev", Ydv)):
    c = Counter(np.minimum(Y.sum(1), 5).tolist())
    print(f"{s}: {len(Y)} docs | statutes per doc: mean {Y.sum(1).mean():.2f} | " +
          ", ".join(f"{'5+' if k == 5 else k}: {100 * v / len(Y):.1f}%" for k, v in sorted(c.items())))
    REPORT[f"{s}_statutes_per_doc"] = {str(k): int(v) for k, v in sorted(c.items())}
co = (Ytr.T.astype(np.int64) @ Ytr.astype(np.int64))
np.fill_diagonal(co, 0)
pairs = [(co[i, j], i, j) for i in range(L) for j in range(i + 1, L) if co[i, j]]
pairs.sort(reverse=True)
print("most common statute pairs (count):", [(LABELS[i], LABELS[j], int(c)) for c, i, j in pairs[:10]])
n = len(Ytr)
lift = sorted(((co[i, j] * n / max(1, ftr[i] * ftr[j]), int(co[i, j]), i, j) for c, i, j in pairs if co[i, j] >= 20), reverse=True)
print("strongest statute pairs (lift, count >= 20):", [(LABELS[i], LABELS[j], round(float(x), 1), c) for x, c, i, j in lift[:8]])
REPORT["top_pairs"] = [(LABELS[i], LABELS[j], int(c)) for c, i, j in pairs[:20]]
words = np.array([len(d["text"].split()) for d in docs["train"]])
print(f"train doc length (words): median {np.median(words):.0f}, 90th pct {np.quantile(words, 0.9):.0f}, max {words.max()}")
pats = {"'section/sec. <number>'": r"\b(?:section|sec\.?|s\.)\s*\d+", "'IPC / Penal Code / CrPC'": r"\b(?:ipc|i\.p\.c|penal code|cr\.?p\.?c|criminal procedure)\b",
        "'Article <number>'": r"\barticle\s*\d+", "mask tokens": r"<mask>|\[mask\]|\bmasked\b"}
REPORT["mentions"] = {}
for name, p in pats.items():
    rx = re.compile(p, re.I)
    share = float(np.mean([bool(rx.search(d["text"])) for d in docs["train"][:5000]]))
    REPORT["mentions"][name] = round(share, 4)
    print(f"train docs mentioning {name}: {100 * share:.1f}% (first 5,000 docs)")
REPORT["train_words"] = {"median": float(np.median(words)), "p90": float(np.quantile(words, 0.9))}
torch.save({"labels": LABELS, **{s: [{"id": d["id"], "text": d["text"], "y": d["y"]} for d in docs[s]] for s in ("train", "dev")}},
           OUT_DIR / "p5_clean.pt")

## 4. Tokenize (T2 for C and D, T1 for A and B) and split train 80 / 20

The split is made once (stratified by each document's rarest statute) and shared by both tokenizers, so all runs see the same held-out documents.

In [ ]:
import legal_d  # noqa: E402
from tokenizers import Tokenizer  # noqa: E402

legal_d.load_tokenizer(T2_PATH)
sp = legal_d.sp
t1 = Tokenizer.from_file(str(T1_PATH))
TOK = {"T2": {"pad": sp.pad_id() if sp.pad_id() >= 0 else 0, "fallback": legal_d.EOS_ID},
       "T1": {"pad": t1.token_to_id("<pad>") if t1.token_to_id("<pad>") is not None else 0,
              "fallback": t1.token_to_id("<eos>") if t1.token_to_id("<eos>") is not None else 0}}


def enc_t2(texts):
    try:
        return sp.encode(texts, out_type=int, num_threads=os.cpu_count() or 4)
    except TypeError:
        return sp.encode(texts, out_type=int)


def enc_t1(texts):
    return [e.ids for e in t1.encode_batch(texts)]


SPLIT = LSI.split_train(docs["train"])
print("split:", {k: len(v) for k, v in SPLIT.items()})
DATA_PATH = {}
for name, enc in (("T2", enc_t2), ("T1", enc_t1)):
    t0 = time.time()
    data = {"pad_id": TOK[name]["pad"], "tokenizer": name, "labels": LABELS, "split": SPLIT}
    for s in ("train", "dev", "test"):
        data[s] = LSI.tokenize_docs(docs[s], enc, TOK[name]["fallback"], LIDX)
    ntok = np.array([d["n_tok"] for d in data["train"]])
    nch = np.array([len(d["chunks"]) for d in data["train"]])
    print(f"{name}: tokenized in {time.time() - t0:.0f}s | tokens per doc: median {np.median(ntok):.0f}, 90th pct "
          f"{np.quantile(ntok, 0.9):.0f} | chunks per doc: mean {nch.mean():.2f}, 1 chunk: {100 * np.mean(nch == 1):.1f}% | "
          f"{100 * np.mean([d['n_win'] > LSI.MAX_CHUNKS for d in data['train']]):.1f}% of docs longer than {LSI.MAX_CHUNKS} chunks (spread)")
    REPORT[f"{name}_tokens"] = {"median": float(np.median(ntok)), "p90": float(np.quantile(ntok, 0.9)), "chunks_mean": float(nch.mean())}
    DATA_PATH[name] = str(OUT_DIR / f"p5_data_{name}.pt")
    torch.save(data, DATA_PATH[name])
    del data
json.dump(REPORT, open(OUT_DIR / "data_report.json", "w"), indent=1)
TEST_META = [{"id": d["id"], "raw_id": d["raw_id"]} for d in docs["test"]]
DEV_IDS = [d["id"] for d in docs["dev"]]
DEV_DOCS = [{"id": d["id"], "y": d["y"]} for d in docs["dev"]]
del docs
gc.collect()

## 5. Self-tests (tiny models, a few seconds)

In [ ]:
import a_model  # noqa: E402
import b_model  # noqa: E402
import c_model  # noqa: E402


def make_tiny(kind):
    if kind == "D":
        cfg = dict(legal_d.MODEL_D_CONFIG, vocab_size=64, d_model=32, n_enc_layer=2, n_dec_layer=1, n_head=4,
                   n_kv_head=2, d_head=8, d_ff=48, max_enc_len=64, max_dec_len=64)
        m = legal_d.build_model(cfg).eval()
        m.dec_blocks = torch.nn.ModuleList()
        return m, 32
    if kind == "C":
        cfg = dict(vocab_size=64, d_model=32, n_layer=2, n_head=4, n_kv_head=2, d_head=8, d_ff=48, max_seq_len=64,
                   qk_norm=True, bias=True, mlp="gelu", tokenizer_id="T2", attn_type="prefix", tied_embeddings=True)
        return c_model.build_model(cfg).eval(), 32
    if kind == "A":
        cfg = dict(vocab_size=64, d_model=32, n_layer=2, n_head=4, n_kv_head=2, d_head=8, d_ff=48, max_seq_len=128)
        return a_model.build_model(cfg).eval(), 32
    if kind == "B":
        cfg = dict(vocab_size=64, d_model=32, n_layer=3, n_head=4, n_kv_head=1, d_head=8, d_ff=48, max_seq_len=128,
                   window=4, global_every=3, qk_norm=True, tokenizer_id="T1", attn_type="local+global", tied_embeddings=True)
        return b_model.build_model(cfg).eval(), 32
    raise KeyError(kind)


LSI.self_test(make_tiny, torch.device("cpu"), kinds=("D", "C", "A", "B"))

## 6. Experiment settings

`budget_min` caps stage-1 training; `est_min` is the whole run (training + encoding + readers) and is only used to skip runs that could not finish in the session.

In [ ]:
COMMON = dict(epochs=2, seed=13, lr_head=1e-3, grad_ckpt=True, val_docs=1000)
EXPERIMENTS = {
    "S1": dict(title="D encoder + LoRMA", model="D", method="lorma", r=16, alpha=32, lr=5e-4,
               micro_chunks=24, accum=2, budget_min=55, infer_bs=64, kd_teacher=True, est_min=110, submit=True),
    "S3": dict(title="C + Houlsby adapters", model="C", method="adapter", bottleneck=64, lr=1e-3,
               micro_chunks=24, accum=2, budget_min=55, infer_bs=48, kd_teacher=True, est_min=110, submit=True),
    "S2": dict(title="B + QLoRA (NF4 4-bit + LoRA)", model="B", method="qlora", r=16, alpha=32, lr=5e-4,
               micro_chunks=12, accum=4, budget_min=60, infer_bs=32, est_min=130, submit=True),
    "S5": dict(title="D encoder + partial FT (top 6 layers)", model="D", method="partial", top_k=6, lr=1e-4, lr_decay=0.85,
               micro_chunks=24, accum=2, budget_min=50, infer_bs=64, est_min=100, submit=True),
    "S6": dict(title="KD student: half of D <- S1 + S3", type="student", model="Ds", student_layers=[0, 2, 4, 6, 8, 10],
               lr=1e-4, micro_docs=6, accum=3, budget_min=60, infer_docs=32, teachers=["S1", "S3"],
               teacher_names="S1 (D + LoRMA) + S3 (C + adapters)", kd_alpha=0.5, kd_T=2.0, est_min=90, submit=True),
    "S4": dict(title="A + prefix tuning", model="A", method="prefix", n_prefix=16, lr=3e-3,
               micro_chunks=12, accum=4, budget_min=60, infer_bs=24, est_min=150, submit=True),
    "R2": dict(title="Student, gold labels only (control)", type="student", model="Ds", student_layers=[0, 2, 4, 6, 8, 10],
               lr=1e-4, micro_docs=6, accum=3, budget_min=60, infer_docs=32, teachers=[], kd_alpha=0.0, est_min=90, submit=False),
    "R1": dict(title="D encoder full fine-tune (reference)", model="D", method="full", lr=3e-5,
               micro_chunks=24, accum=2, budget_min=45, infer_bs=64, est_min=95, submit=False),
}
RUN_ORDER = [("S1", []), ("S3", []), ("S2", []), ("S5", []), ("S6", ["S1", "S3"]), ("S4", []), ("R2", []), ("R1", [])]
TIME_LIMIT_MIN = 11 * 60          # Kaggle stops a session at 12 h: never start a run that would end after 11 h
CFG = {"work_dir": str(WORK_DIR), "data": DATA_PATH, "ckpt": CKPT, "out_dir": str(OUT_DIR),
       "common": COMMON, "experiments": EXPERIMENTS}
CONFIG_PATH = WORK_DIR / "lsi_config.json"
json.dump(CFG, open(CONFIG_PATH, "w"), indent=1)
for name, e in EXPERIMENTS.items():
    print(f"{name}: {e['title']}")

## 7. Run (two at a time, one per GPU; the TF-IDF reference runs on the CPU alongside)

In [ ]:
# reference without pretraining, on CPU while the GPUs train
tfidf_log = open(OUT_DIR / "tfidf.log", "w")
tfidf = subprocess.Popen([sys.executable, str(WORK_DIR / "lsi_tfidf.py"), str(OUT_DIR / "p5_clean.pt"),
                          str(OUT_DIR / "tfidf.json"), str(WORK_DIR)], stdout=tfidf_log, stderr=subprocess.STDOUT)


def stream(proc, tag):
    with open(OUT_DIR / f"log_{tag}.txt", "w") as f:
        for line in proc.stdout:
            f.write(line)
            print(line if line.startswith(f"[{tag}]") else f"[{tag}] {line}", end="", flush=True)


queue, running, codes = list(RUN_ORDER), {}, {}
slots = list(range(N_GPUS))[:2] if N_GPUS else [-1]
while queue or running:
    for slot in slots:
        if slot in running:
            continue
        ready = [q for q in queue if all(d in codes for d in q[1])]
        if not ready:
            continue
        name, deps = ready[0]
        queue.remove(ready[0])
        if any(codes.get(d) != 0 for d in deps):
            print(f">>> skipping {name}: {deps} did not all finish", flush=True)
            codes[name] = -1
            continue
        used = (time.time() - NB_T0) / 60
        if used + EXPERIMENTS[name]["est_min"] > TIME_LIMIT_MIN:
            print(f">>> skipping {name}: {used:.0f} min used, it needs ~{EXPERIMENTS[name]['est_min']} more", flush=True)
            codes[name] = -2
            continue
        env = dict(os.environ, PYTHONUNBUFFERED="1", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
        proc = subprocess.Popen([sys.executable, str(WORK_DIR / "lsi_train.py"), str(CONFIG_PATH), name, str(slot)],
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
        th = threading.Thread(target=stream, args=(proc, name), daemon=True)
        th.start()
        running[slot] = (name, proc, th)
        print(f">>> started {name} ({EXPERIMENTS[name]['title']}) on {'GPU ' + str(slot) if slot >= 0 else 'CPU'} "
              f"({(time.time() - NB_T0) / 60:.0f} min into the notebook)", flush=True)
    for slot, (name, proc, th) in list(running.items()):
        if proc.poll() is not None:
            th.join()
            codes[name] = proc.returncode
            print(f">>> {name} finished with code {proc.returncode} ({(time.time() - NB_T0) / 60:.0f} min into the notebook)", flush=True)
            del running[slot]
    time.sleep(5)
print("Exit codes:", codes)
try:
    tfidf.wait(timeout=max(60, (TIME_LIMIT_MIN + 30) * 60 - (time.time() - NB_T0)))
except subprocess.TimeoutExpired:
    tfidf.kill()
tfidf_log.close()
print(open(OUT_DIR / "tfidf.log").read()[-1500:])

## 8. S7: ensemble of the submitted runs

In [ ]:
# S7: mean probability of every finished submission run, thresholds re-fitted the same honest way
DATA_T2 = torch.load(DATA_PATH["T2"], weights_only=False)
Yd = LSI.label_matrix(DATA_T2["dev"], L)
NEVER = LSI.unseen(DATA_T2)
members = [n for n, e in EXPERIMENTS.items() if e["submit"] and (OUT_DIR / n / "scores.npz").exists()]
sc = {n: np.load(OUT_DIR / n / "scores.npz") for n in members}
for n in members:
    assert list(sc[n]["dev_ids"]) == DEV_IDS and list(sc[n]["test_ids"]) == [t["id"] for t in TEST_META], f"{n}: id order differs"
    assert list(sc[n]["labels"]) == LABELS, f"{n}: statute order differs"
print("S7 members:", members)
if len(members) >= 2:
    S7 = OUT_DIR / "S7"
    S7.mkdir(exist_ok=True)
    dev_s, test_s = (LSI.blend(*[sc[n][k] for n in members]) for k in ("dev", "test"))
    thr, cal = LSI.calibrate(dev_s, Yd, 13, never=NEVER)
    Pd, Pt = LSI.apply_thresholds(dev_s, thr), LSI.apply_thresholds(test_s, thr)
    rep = LSI.multilabel_report(Yd, Pd)
    LSI.write_submission(S7, LSI.submission_rows(TEST_META, Pt, LABELS))
    np.savez_compressed(S7 / "scores.npz", dev=dev_s.astype(np.float32), test=test_s.astype(np.float32),
                        dev_ids=np.array(DEV_IDS), test_ids=np.array([t["id"] for t in TEST_META]), labels=np.array(LABELS))
    info = {"exp": {"name": "S7", "title": "Ensemble " + " + ".join(members)}, "members": members, "dev_report": rep,
            "dev_report_2fold": cal["strategy_2fold"][cal["chosen_strategy"]],
            "dev_macro_ap": LSI.macro_ap(Yd, dev_s), "calibration": cal, "threshold": thr.tolist(),
            "analysis": LSI.analysis(Yd, Pd, dev_s, thr, DATA_T2, DATA_T2["dev"], S7)}
    json.dump(info, open(S7 / "metrics.json", "w"), indent=1, default=str)
    print(f"S7 ensemble: dev macro-F1 {info['dev_report_2fold']['macro_f1']:.4f} with 2-fold thresholds, {rep['macro_f1']:.4f} with "
          f"thresholds fitted on all dev (P {rep['macro_precision']:.4f} R {rep['macro_recall']:.4f}) | "
          f"micro-F1 {rep['micro_f1']:.4f} | subset acc {rep['subset_accuracy']:.4f} | thresholds {cal['chosen_strategy']}")

## 9. Results, analysis tables, submissions

In [ ]:
rows, by = [], {}
for name, _ in RUN_ORDER + [("S7", [])]:
    mpath = OUT_DIR / name / "metrics.json"
    if not mpath.exists():
        print(f"{name}: no metrics (see its log above)")
        continue
    m = json.load(open(mpath))
    e = EXPERIMENTS.get(name, {"title": m["exp"]["title"], "model": "-", "submit": True})
    rep = m["dev_report"]
    r = {"run": name, "title": e["title"], "model": e["model"],
         "technique": e.get("method", "ensemble" if name == "S7" else ("distillation" if e.get("teachers") else "student, gold only")),
         "backbone_params": m.get("n_backbone", 0), "trainable": m.get("n_trainable_backbone", 0), "trainable_pct": m.get("trainable_pct", 0),
         "storage_mb": m.get("backbone_storage_mb", 0), "dev_macro_f1_2fold": m["dev_report_2fold"]["macro_f1"],
         "dev_macro_f1": rep["macro_f1"], "dev_macro_p": rep["macro_precision"],
         "dev_macro_r": rep["macro_recall"], "dev_micro_f1": rep["micro_f1"], "dev_subset_acc": rep["subset_accuracy"],
         "dev_macro_ap": m["dev_macro_ap"], "statutes_per_doc": rep["avg_predicted_per_doc"], "reader": m.get("chosen_reader", "-"),
         "thresholds": m["calibration"]["chosen_strategy"],
         "minutes": round(m.get("stage1_minutes", m.get("train_minutes", 0)) + m.get("encode_minutes", 0), 1),
         "peak_gb": m.get("peak_mem_train_gb", 0), "docs_per_s": m.get("infer_docs_per_s", 0), "submit": e["submit"]}
    rows.append(r)
    by[name] = (r, m)
assert rows, "No run finished -- read the error lines above."
with open(OUT_DIR / "runs.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0]))
    w.writeheader()
    w.writerows(rows)
print("macro-F1 (2-fold) = thresholds fitted on one half of dev, scored on the other (honest, compare runs on this);\n"
      "macro-F1 (fit) = thresholds fitted on all of dev (what the submission uses; optimistic on dev).")
print(f"{'run':4s} {'title':38s} {'trainable':>12s} {'%':>7s} {'MB':>6s} {'F1 2fold':>8s} {'F1 fit':>7s} {'P':>6s} {'R':>6s} {'microF1':>8s} "
      f"{'subset':>7s} {'mAP':>6s} {'min':>6s} {'GB':>5s} {'doc/s':>6s}")
for r in rows:
    print(f"{r['run']:4s} {r['title'][:38]:38s} {r['trainable']:>12,} {r['trainable_pct']:7.2f} {r['storage_mb']:6.0f} "
          f"{r['dev_macro_f1_2fold']:8.4f} {r['dev_macro_f1']:7.4f} {r['dev_macro_p']:6.3f} {r['dev_macro_r']:6.3f} {r['dev_micro_f1']:8.4f} {r['dev_subset_acc']:7.4f} "
          f"{r['dev_macro_ap']:6.3f} {r['minutes']:6.1f} {r['peak_gb']:5.1f} {r['docs_per_s']:6.1f}")

for name, (r, m) in by.items():
    print(f"\n===== {name}: {r['title']} =====")
    if "readers" in m:
        print(f"readers (chosen by validation macro-AP: {m['chosen_reader']}; dev numbers are for the report only):")
        for x in m["readers"]:
            print(f"  {x['reader']:32s} val mAP {x['val_macro_ap']:.4f} | dev mAP {x['dev_macro_ap']:.4f} | dev macro-F1 {x['dev_macro_f1']:.4f} "
                  f"| micro-F1 {x['dev_micro_f1']:.4f}  ({x['note']})")
    print("threshold strategies (2-fold dev macro-F1):",
          {k: v["macro_f1"] for k, v in m["calibration"]["strategy_2fold"].items()}, "-> chosen", m["calibration"]["chosen_strategy"])
    a = m["analysis"]
    print("by statute frequency:", a["by_statute_frequency"])
    print("by number of gold statutes:", a["by_gold_count"])
    print("by document length:", a["by_length"])
    print("most missed:", a["most_missed"], "| most over-predicted:", a["most_over_predicted"])
    print("missed -> predicted instead:", a["missed_vs_predicted_instead"])
    print(f"statutes never predicted: {a['statutes_never_predicted']} of {a['statutes_in_dev']} in dev")
    if m.get("nf4_relative_weight_error") is not None:
        print(f"QLoRA: 4-bit weights are {100 * m['nf4_relative_weight_error']:.1f}% away from the Phase-1 weights (relative Frobenius error)")
    if "stage1_history" in m:
        print("stage 1:", [(h["progress"], h["val_macro_ap"]) for h in m["stage1_history"]])

# distillation: teachers vs student vs the same student without distillation
if all(n in by for n in ("S1", "S3")):
    t = [np.load(OUT_DIR / n / "scores.npz") for n in ("S1", "S3")]
    ts = LSI.blend(t[0]["dev"], t[1]["dev"])
    thr, cal = LSI.calibrate(ts, Yd, 13, never=NEVER)
    tr = cal["strategy_2fold"][cal["chosen_strategy"]]
    tp = by["S1"][1]["n_backbone"] + by["S3"][1]["n_backbone"]
    tsp = 1 / sum(1 / max(1e-9, by[n][0]["docs_per_s"]) for n in ("S1", "S3"))
    print(f"\nTeacher ensemble S1 + S3: dev macro-F1 (2-fold) {tr['macro_f1']:.4f} | micro-F1 {tr['micro_f1']:.4f} | {tp / 1e6:.1f}M parameters, "
          f"{tsp:.1f} dev docs/s (all chunks)")
    for n in ("S6", "R2"):
        if n in by:
            r, m = by[n]
            print(f"  {n} {r['title']}: {m['n_backbone'] / 1e6:.1f}M ({100 * m['n_backbone'] / tp:.0f}% of the teachers), {r['docs_per_s']:.1f} "
                  f"docs/s (<= {LSI.KD_CHUNKS} chunks), dev macro-F1 (2-fold) {r['dev_macro_f1_2fold']:.4f}, micro-F1 {r['dev_micro_f1']:.4f}")
if (OUT_DIR / "tfidf.json").exists():
    tf = json.load(open(OUT_DIR / "tfidf.json"))
    print(f"\nTF-IDF + logistic regression (no pretraining): dev macro-F1 (2-fold) {tf['dev_report_2fold']['macro_f1']:.4f} | "
          f"micro-F1 {tf['dev_report']['micro_f1']:.4f} | macro-AP {tf['dev_macro_ap']:.4f}")

with zipfile.ZipFile(OUT_DIR / "report_files.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT_DIR.rglob("*")):
        if f.is_file() and f.suffix in (".json", ".csv", ".txt", ".log") and f.name != "prediction.json":
            z.write(f, arcname=str(f.relative_to(OUT_DIR)))
print(f"\nAll tables for the report: {OUT_DIR / 'report_files.zip'}")
print("\nSubmit these (Output tab):")
for name in [n for n, e in EXPERIMENTS.items() if e["submit"]] + ["S7"]:
    zf = OUT_DIR / name / "submission.zip"
    if zf.exists():
        pred = json.loads(zipfile.ZipFile(zf).read("prediction.json"))
        assert len(pred) == len(TEST_META) and all(p["pred"] for p in pred)
        print(f"  {name} ({by[name][0]['title'] if name in by else ''}): {zf}  "
              f"[dev macro-F1 {by[name][0]['dev_macro_f1_2fold'] if name in by else float('nan'):.4f} (2-fold)]")
    else:
        print(f"  {name}: MISSING")